If SNP B is causal, SNP A and SNP C may also appear significant because they are inherited together.

Therefore:

GWAS answers:

> Which genomic region is associated with the trait?

Fine-mapping answers:

> Which specific variant is most likely causing the association?


### Why GWAS Alone Is Not Enough

Suppose a GWAS identifies a significant locus on chromosome 22.

The Manhattan plot may show a strong peak:

```text
          *
         ***
       *******
     ***********
```

The peak could contain:

* 10 SNPs
* 100 SNPs
* 1000 SNPs

all showing nearly identical p-values.

GWAS cannot determine which SNP is truly causal because many neighboring SNPs are correlated through LD.

For example:

```text
rs1
rs2
rs3
rs4
rs5
```

All five SNPs may have nearly identical p-values.

Fine-mapping attempts to separate these signals and determine which SNP has the strongest evidence for causality.




### What Is the Goal of Fine-Mapping?

Fine-mapping attempts to estimate:

```text
P(SNP is causal | observed data)
```

for every SNP in a genomic region.

Instead of reporting only p-values, fine-mapping reports probabilities.

These probabilities are called:

#### Posterior Inclusion Probabilities (PIPs)

For example:

```text
rs1 : 0.01
rs2 : 0.04
rs3 : 0.88
rs4 : 0.05
rs5 : 0.02
```

Interpretation:

```text
rs3 has an 88% probability of being causal
```

and is therefore the strongest candidate variant.




### GWAS Versus Fine-Mapping

GWAS is designed to detect associated regions.

Fine-mapping is designed to identify likely causal variants within those regions.

GWAS focuses on:

* p-values
* association testing
* genome-wide discovery

Fine-mapping focuses on:

* posterior probabilities
* causal inference
* variant prioritization

In modern statistical genetics, GWAS is often considered the first step, while fine-mapping is the second step.




### Why Fine-Mapping Matters

Imagine a schizophrenia GWAS identifies a significant locus containing 200 SNPs.

Experimental validation of 200 variants would be extremely expensive.

Fine-mapping may reduce the candidate list to:

```text
3 SNPs
```

or even

```text
1 SNP
```

This dramatically reduces the cost and complexity of downstream biological experiments.

Fine-mapping is therefore essential for:

* Functional genomics
* Drug target discovery
* eQTL interpretation
* Colocalization analyses
* Precision medicine

---


### The Modern Statistical Genetics Pipeline

Most modern studies follow a workflow similar to:

```text
Genotypes
     ↓
GWAS
     ↓
Significant Locus
     ↓
LD Matrix
     ↓
Fine-Mapping
     ↓
Posterior Inclusion Probabilities
     ↓
Credible Sets
     ↓
eQTL Analysis
     ↓
Colocalization
     ↓
Candidate Gene
```

This pipeline forms the backbone of many large-scale genetics projects including:

* UK Biobank
* FinnGen
* Psychiatric Genomics Consortium
* GTEx

---


### What Information Does Fine-Mapping Need?

The first question in any fine-mapping study is:

```text
What type of data is available?
```

There are three common situations.


#### Scenario 1: Individual-Level Data

You have access to:

```text
Genotypes
Phenotypes
```

for every participant.

This is the ideal scenario because LD can be computed directly from the study samples.

---

#### Scenario 2: Single-Cohort GWAS Summary Statistics

You only have:

```text
GWAS summary statistics
```

such as:

* beta
* standard error
* p-value

but no individual-level genotypes.

In this case, an external LD reference panel is required.

Examples include:

* 1000 Genomes
* UK Biobank
* TOPMed

---

#### Scenario 3: Meta-Analysis GWAS

You only have meta-analysis summary statistics from multiple cohorts.

This is the most challenging situation because LD differs across cohorts.

Specialized methods such as:

* FastMap
* CARMA
* SLALOM

are often required.

---


### The Central Role of Linkage Disequilibrium

Linkage Disequilibrium (LD) is the most important concept in fine-mapping.

Suppose two SNPs have:

```text
r² = 0.95
```

They are almost perfectly correlated.

Statistically, it becomes difficult to distinguish which SNP is causal.

High LD leads to:

```text
Large credible sets
Low resolution
```

Low LD leads to:

```text
Small credible sets
High resolution
```

This is why ancestry-matched LD reference panels are extremely important.

---

### A Simple Numerical Example

Consider five SNPs:

```python
import pandas as pd

results = pd.DataFrame({
    "SNP":["rs1","rs2","rs3","rs4","rs5"],
    "PIP":[0.01,0.04,0.88,0.05,0.02]
})

results
```

Output:

```text
rs1  0.01
rs2  0.04
rs3  0.88
rs4  0.05
rs5  0.02
```

The most likely causal SNP is:

```text
rs3
```

because it has the highest Posterior Inclusion Probability.

---



### Key Concepts Introduced in This Tutorial

Before learning specific fine-mapping methods, you should understand:

* Linkage Disequilibrium (LD)
* Causal Variants
* Posterior Inclusion Probability (PIP)
* Credible Sets
* LD Reference Panels
* Summary Statistics
* Individual-Level Data

These concepts form the foundation of all modern fine-mapping approaches.

---

### Key Takeaways

Fine-mapping is the process of identifying likely causal variants within a GWAS locus. Because neighboring SNPs are often correlated through linkage disequilibrium, GWAS alone cannot determine which variant is responsible for an association signal. Fine-mapping combines GWAS statistics with LD information and Bayesian inference to assign probabilities of causality to variants. The primary outputs are Posterior Inclusion Probabilities (PIPs) and Credible Sets. Fine-mapping serves as the bridge between GWAS discovery and downstream analyses such as eQTL mapping, colocalization, functional validation, and drug target identification.

---

### Part 2: Linkage Disequilibrium, Posterior Inclusion Probabilities (PIPs), and Credible Sets

Before we can understand methods such as ABF, SuSiE, FINEMAP, CARMA, or FastMap, we must understand three foundational concepts:

1. Linkage Disequilibrium (LD)
2. Posterior Inclusion Probabilities (PIPs)
3. Credible Sets

These concepts form the mathematical backbone of statistical fine-mapping.

---

### What Is Linkage Disequilibrium (LD)?

Linkage Disequilibrium (LD) measures the correlation between genetic variants.

When two SNPs are inherited together more often than expected by chance, they are said to be in LD.

For example:

```text
SNP1 -------- SNP2 -------- SNP3
```

Suppose:

```text
r²(SNP1,SNP2) = 0.95
```

This means:

* Knowing SNP1 almost perfectly predicts SNP2.
* Their genotypes are highly correlated.
* They tend to be inherited together.

---


### Why Does LD Exist?

During reproduction, chromosomes undergo recombination.

However, nearby variants are less likely to be separated.

Therefore:

```text
Nearby SNPs
      ↓
Inherited together
      ↓
Correlation
      ↓
LD
```

The closer two SNPs are physically located, the stronger their LD tends to be.

---

### LD Measures

Two common measures are:

## D'

Measures historical recombination.

Range:

```text
0 → 1
```

---

## r²

Measures correlation.

Range:

```text
0 → 1
```

Interpretation:

```text
r² = 0
No correlation
```

```text
r² = 1
Perfect correlation
```

Fine-mapping primarily uses:

```text
r²
```

---


### Example LD Matrix

Consider four SNPs.

```python
import pandas as pd

ld = pd.DataFrame(
    [
        [1.00,0.92,0.15,0.05],
        [0.92,1.00,0.18,0.07],
        [0.15,0.18,1.00,0.85],
        [0.05,0.07,0.85,1.00]
    ],
    columns=["rs1","rs2","rs3","rs4"],
    index=["rs1","rs2","rs3","rs4"]
)

ld
```

Output:

```text
      rs1   rs2   rs3   rs4
rs1  1.00  0.92  0.15  0.05
rs2  0.92  1.00  0.18  0.07
rs3  0.15  0.18  1.00  0.85
rs4  0.05  0.07  0.85  1.00
```

Interpretation:

```text
rs1 and rs2 form one LD block
rs3 and rs4 form another LD block
```

---


### Why LD Causes Problems in GWAS

Suppose rs1 is truly causal.

Because rs2 has:

```text
r² = 0.92
```

both SNPs may show significant p-values.

Example:

```text
rs1   p = 1e-10
rs2   p = 2e-10
```

GWAS cannot distinguish:

```text
Which SNP is causal?
```

Both appear significant.

This is the central problem that fine-mapping attempts to solve.

---

### The Fine-Mapping Perspective

Instead of asking:

```text
Which SNP has the smallest p-value?
```

Fine-mapping asks:

```text
Which SNP is most likely causal?
```

This distinction is extremely important.

A SNP with the smallest p-value is not always the causal SNP.

---



### Posterior Inclusion Probability (PIP)

The primary output of modern fine-mapping methods is the:

#### Posterior Inclusion Probability

Mathematically:

```text
PIP = P(SNP is causal | data)
```

PIP values range from:

```text
0 → 1
```

Interpretation:

```text
0
Not likely causal
```

```text
1
Almost certainly causal
```

---


### Example PIPs

Suppose fine-mapping returns:

```python
import pandas as pd

pip = pd.DataFrame({
    "SNP":["rs1","rs2","rs3","rs4"],
    "PIP":[0.03,0.07,0.82,0.08]
})

pip
```

Output:

```text
rs1  0.03
rs2  0.07
rs3  0.82
rs4  0.08
```

Interpretation:

```text
rs3 has an 82% probability
of being causal
```


### PIPs Are Not p-values

Many beginners confuse PIPs with p-values.

They are completely different.

##### p-value

Measures:

```text
Evidence against the null hypothesis
```

##### PIP

Measures:

```text
Probability of causality
```

Example:

```text
p-value = 1e-20
```

does NOT mean

```text
99.999999999999999% chance causal
```

That interpretation is incorrect.


### Multiple Causal Variants

Many loci contain more than one causal SNP.

Example:

```text
Variant 20
Variant 39
Variant 68
```

All may influence the same trait.

This is exactly why methods such as SuSiE were developed. The workshop practical simulates three causal variants in the same locus and shows that a locus may contain multiple independent signals. 

---

### What Is a Credible Set?

A Credible Set is a group of variants that collectively contain the causal SNP with high probability.

Most commonly:

```text
95% Credible Set
```

Meaning:

```text
There is a 95% probability
that the causal SNP is inside
this set.
```

---

### Example Credible Set

Suppose:

```text
rs1  PIP = 0.60
rs2  PIP = 0.25
rs3  PIP = 0.08
rs4  PIP = 0.04
rs5  PIP = 0.03
```

Cumulative probabilities:

```text
rs1 = 0.60

rs1 + rs2 = 0.85

rs1 + rs2 + rs3 = 0.93

rs1 + rs2 + rs3 + rs4 = 0.97
```

The 95% credible set becomes:

```text
rs1
rs2
rs3
rs4
```



because together they exceed 95%.

---

### Small Versus Large Credible Sets

Small credible set:

```text
rs25
```

Excellent resolution.

---

Medium credible set:

```text
rs25
rs26
rs27
```

Reasonable resolution.

---

Large credible set:

```text
50 SNPs
```

Poor resolution.

---

### What Determines Credible Set Size?

Several factors influence fine-mapping resolution.

#### Sample Size

Larger sample sizes:

```text
Higher power
```

```text
Smaller credible sets
```

The workshop demonstrates that increasing sample size improves PIPs and fine-mapping resolution. 

---

#### LD Structure

High LD:

```text
Large credible sets
```

Low LD:

```text
Small credible sets
```

The workshop also shows that increasing LD makes causal variants harder to resolve. 

---

#### Effect Size

Larger effects:

```text
Higher PIPs
```

Smaller effects:

```text
Lower PIPs
```

---

### Visual Interpretation

Ideal situation:

```text
rs39
PIP = 0.98
```

Credible Set:

```text
{rs39}
```

Excellent.

---

Difficult situation:

```text
rs39 = 0.20
rs40 = 0.19
rs41 = 0.18
rs42 = 0.17
rs43 = 0.16
```

Credible Set:

```text
5 variants
```

Less certainty.

---

### Why Fine-Mapping Uses Bayesian Statistics

Bayesian methods naturally answer:

```text
How likely is each SNP
to be causal?
```

which is exactly the scientific question we care about.

This is why nearly all modern fine-mapping methods:

* ABF
* SuSiE
* FINEMAP
* CARMA
* CAVIAR

are Bayesian methods.

---

### Key Takeaways

Linkage Disequilibrium is the correlation structure among nearby variants and is the primary reason GWAS cannot identify causal variants directly. Fine-mapping uses LD information together with GWAS statistics to estimate Posterior Inclusion Probabilities (PIPs), which represent the probability that a variant is causal. Variants with high PIPs are prioritized for follow-up studies. Because uncertainty remains, fine-mapping methods typically report Credible Sets, collections of variants that together contain the causal variant with high probability. Sample size, effect size, and LD structure strongly influence PIPs and credible set sizes.

---

### Part 3: Bayesian Fine-Mapping and Approximate Bayes Factors (ABF)

In the previous section, we learned that GWAS identifies associated regions but cannot reliably determine which variant is causal because of Linkage Disequilibrium (LD).

The next question becomes:

```text
How can we assign probabilities of causality to SNPs?
```

One of the earliest and most influential solutions is:

### Approximate Bayes Factors (ABF)

ABF forms the foundation of many modern fine-mapping methods and remains widely used today.

The ISG workshop uses Jon Wakefield's Approximate Bayes Factor approach before introducing SuSiE. 

---

### Why Move Beyond p-values?

Suppose a GWAS reports:

```text
rs1   p = 1×10⁻⁸
rs2   p = 2×10⁻⁸
rs3   p = 5×10⁻⁸
```

All three SNPs are significant.

But p-values do not tell us:

```text
Which SNP is causal?
```

They only tell us:

```text
How unlikely the data would be
if no association existed.
```

Fine-mapping requires something different:

```text
Probability that a SNP is causal.
```

This is where Bayesian statistics becomes useful.

---



### Frequentist Versus Bayesian Thinking

### Frequentist Approach

The null hypothesis is:

```text
β = 0
```

We calculate:

```text
P(Data | Null)
```

which produces a p-value.

---

### Bayesian Approach

We compare:

```text
Model 1:
SNP is causal
```

versus

```text
Model 0:
SNP is not causal
```

and calculate:

```text
P(Causal | Data)
```

This is exactly the quantity we want.

---

### What Is a Bayes Factor?

A Bayes Factor compares two competing models.

Mathematically:

```text
BF =
P(Data | Causal Model)
----------------------
P(Data | Null Model)
```

Interpretation:

```text
BF = 1
No preference
```

```text
BF = 10
Data are 10 times more likely
under the causal model
```

```text
BF = 100
Strong evidence for causality
```

---

### Why "Approximate" Bayes Factor?

Computing exact Bayes Factors can be computationally expensive.

Wakefield proposed an approximation that requires only:

* Effect size estimate (β)
* Standard error (SE)

These quantities are available in almost every GWAS summary statistics file.

Therefore:

```text
Summary Statistics
      ↓
ABF
      ↓
Posterior Probability
```

No individual-level data are required.

---

### Inputs Required for ABF

For each SNP we need:

```text
Beta
```

Example:

```text
β = 0.12
```

and

```text
SE = 0.03
```

From these values we compute:

```text
Z = β / SE
```

---

### Example

Suppose:

```python
beta = 0.12
se = 0.03

z = beta / se

print(z)
```

Output:

```text
4.0
```

A large absolute Z-score indicates stronger evidence for association.

---

### Wakefield's Approximate Bayes Factor

The workshop implements the following ABF calculation. 

First define:

```text
V = SE²
```

Then choose a prior variance:

```text
W
```

A common value:

```text
W = 0.04
```

Next calculate:

```text
r = W / (W + V)
```

The log Bayes Factor becomes:

```text
lBF =
(log(1-r) + rZ²)/2
```

This transforms GWAS statistics into Bayesian evidence.

---

### Understanding the Prior Variance W

The parameter:

```text
W
```

represents our prior belief about effect sizes.

Small W:

```text
Expect small effects
```

Large W:

```text
Allow larger effects
```

Typical fine-mapping analyses use values between:

```text
0.01 and 0.04
```

---

### From Bayes Factors to Posterior Probabilities

After computing Bayes Factors for every SNP:

```text
rs1   BF = 3
rs2   BF = 50
rs3   BF = 10
```

we normalize them.

Suppose:

```text
Total BF = 63
```

Then:

```text
PIP(rs1) = 3/63 = 0.048
PIP(rs2) = 50/63 = 0.794
PIP(rs3) = 10/63 = 0.159
```

These are the Posterior Inclusion Probabilities.

---

### A Complete ABF Example

```python
import numpy as np
import pandas as pd

beta = np.array([0.05,0.15,0.09])
se = np.array([0.03,0.03,0.03])

W = 0.04

z = beta/se
V = se**2

r = W/(W+V)

lbf = (np.log(1-r) + r*(z**2))/2

bf = np.exp(lbf)

pip = bf / bf.sum()

results = pd.DataFrame({
    "Beta":beta,
    "SE":se,
    "Z":z,
    "BF":bf,
    "PIP":pip
})

results
```

---

### Interpreting the Results

Suppose the output is:

```text
SNP1  PIP = 0.05
SNP2  PIP = 0.82
SNP3  PIP = 0.13
```

Interpretation:

```text
SNP2 is the strongest
causal candidate.
```

---

### The Single-Causal-Variant Assumption

ABF makes a very important assumption:

```text
Only one causal variant
exists in the region.
```

This assumption simplifies the mathematics.

However, many real loci contain:

```text
2
3
5
or more
causal variants
```

This limitation motivated the development of more advanced methods such as:

* SuSiE
* FINEMAP
* CAVIAR
* CARMA

---

### Why ABF Sometimes Fails

Imagine:

```text
Variant 20 is causal
Variant 39 is causal
Variant 68 is causal
```

ABF assumes:

```text
Only one can be causal.
```

Therefore:

```text
Signal splitting occurs
```

and the true signals may receive lower PIPs.

The workshop demonstrates exactly this scenario by simulating three causal variants in the same locus. 

---

### Advantages of ABF

ABF is still popular because it is:

* Fast
* Easy to compute
* Requires only summary statistics
* Does not require an LD matrix
* Computationally efficient

This makes it valuable for large GWAS datasets.

---

### Limitations of ABF

ABF assumes:

```text
One causal variant per locus
```

which is often unrealistic.

Other limitations:

* Cannot separate multiple signals
* Can be confused by strong LD
* Lower resolution in complex regions

These limitations motivated modern methods such as SuSiE.

---

### ABF in Colocalization

ABF is also used in:

```text
coloc.abf
```

for testing whether GWAS and eQTL signals share the same causal variant.

The colocalization workshop specifically notes that coloc.abf requires only summary statistics and assumes at most one causal variant per locus. 

This makes ABF an important building block in statistical genetics.

---

### Visual Summary

```text
GWAS Summary Statistics
           ↓
      Compute Z
           ↓
      Compute ABF
           ↓
      Bayes Factors
           ↓
      Normalize
           ↓
      PIPs
           ↓
      Credible Sets
```

---

### Key Takeaways

Approximate Bayes Factors (ABF) were among the first practical Bayesian fine-mapping methods. They use GWAS summary statistics to estimate evidence that each SNP is causal. By normalizing Bayes Factors across SNPs, ABF produces Posterior Inclusion Probabilities (PIPs), which quantify the probability that a variant is causal. ABF is computationally efficient and requires only summary statistics, making it useful when LD information is unavailable. However, its major limitation is the assumption that only one causal variant exists within a locus. Because many real loci contain multiple independent signals, more advanced methods such as SuSiE were developed to overcome this limitation.

---


### Part 4: SuSiE (Sum of Single Effects)

Approximate Bayes Factors (ABF) introduced an important idea:

```text 
Estimate the probability
that each SNP is causal.
```

However, ABF assumes:

```text
One causal variant
per locus
```

Unfortunately, real biological loci rarely behave this way.

Many GWAS regions contain:

```text 
2
3
5
or more
independent causal variants
```

To solve this problem, modern fine-mapping methods such as:

* SuSiE
* FINEMAP
* CARMA

were developed.

Among these, SuSiE has become one of the most widely used methods in statistical genetics.

The ISG workshop uses SuSiE as the primary modern fine-mapping method. 

---

### What Does SuSiE Mean?

SuSiE stands for:

#### Sum of Single Effects

The key idea is surprisingly simple.

Instead of assuming:

```text 
One causal SNP
```

SuSiE assumes:

```text 
Several causal SNPs
```

and models the phenotype as the sum of multiple independent effects.

Mathematically:

```text 
Phenotype

=
Effect 1
+
Effect 2
+
Effect 3
+
...
```

Hence:

```text
Sum of Single Effects
```

---

### Why Was SuSiE Developed?

Imagine a locus containing:

```text 
Variant 20
Variant 39
Variant 68
```

Suppose all three are truly causal.

ABF attempts to explain the locus using:

```text 
One SNP
```

SuSiE attempts to explain the locus using:

```text 
Multiple SNPs
```

This is much closer to biological reality.

The workshop simulation deliberately contains three causal variants in the same region. 

---

### The Core SuSiE Model

SuSiE models the phenotype as:

```text 
y = Xb + e
```

where:

```text 
y
```

is the phenotype,

```text 
X
```

is the genotype matrix,

```text 
b
```

contains SNP effects,

and

```text 
e
```

is random noise.

---

### The Single-Effect Idea

Instead of fitting all SNP effects simultaneously, SuSiE decomposes the model into:

```text 
Effect 1
Effect 2
Effect 3
...
```

Each effect identifies one potential causal signal.

For example:

```text 
Signal 1 → Variant 39

Signal 2 → Variant 20

Signal 3 → Variant 68
```

The final model combines them.

---

### The Parameter L

A very important SuSiE parameter is:

```text 
L
```

which specifies the maximum number of causal signals.

Example:

```text 
L = 4
```

means:

```text 
Allow up to
4 independent signals
```

The workshop uses:

```r
allowed_causal <- L + 1
```

and then fits:

```r
susie_rss(
  ...
  L = allowed_causal
)
```

to search for multiple independent signals. 

---

### What Does SuSiE Produce?

SuSiE returns two important outputs:

#### Posterior Inclusion Probabilities

```text 
PIP
```

for every SNP.

---

#### Credible Sets

Groups of SNPs likely to contain causal variants.

Example:

```text 
CS1:
rs39
```

```text 
CS2:
rs20
```

Each credible set corresponds to a distinct signal.

---

### Example: Three Causal Variants

Suppose the truth is:

```text 
Variant 20
Variant 39
Variant 68
```

ABF may identify:

```text 
Variant 39 only
```

because it assumes one signal.

SuSiE can identify:

```text 
Signal 1 → Variant 20

Signal 2 → Variant 39

Signal 3 → Variant 68
```

This is the major advantage of SuSiE.

---

### Running SuSiE in R

The workshop uses:

```r
library(susieR)

susie_results <- susie_rss(
    bhat = GWAS_df$beta_marginal,
    shat = GWAS_df$se_marginal,
    n = N,
    R = in_sample_LD,
    var_y = var(y),
    L = 4,
    estimate_residual_variance = TRUE
)
```

Notice the inputs:

```text 
Beta estimates
Standard errors
Sample size
LD matrix
Phenotype variance
```

Unlike ABF:

```text 
SuSiE requires LD
```

because it explicitly models multiple correlated signals.



---

### Why Does SuSiE Need an LD Matrix?

Suppose:

```text 
rs39
```

and

```text 
rs40
```

have:

```text 
r² = 0.95
```

SuSiE needs LD information to determine whether:

```text 
One signal
```

or

```text 
Two independent signals
```

exist.

Without LD, this distinction becomes impossible.

---

### Posterior Inclusion Probabilities in SuSiE

Like ABF, SuSiE reports PIPs.

Example:

```text 
rs20  0.95

rs39  0.98

rs68  0.92
```

Interpretation:

```text 
All three SNPs
are likely causal.
```

This is much more informative than a single p-value.

---

### Credible Sets in SuSiE

Suppose SuSiE reports:

```text 
CS1 = {rs20}

CS2 = {rs39}

CS3 = {rs68}
```

This is excellent resolution.

---

More commonly:

```text
CS1 =
rs20
rs21
rs22
```

meaning:

```text 
One of these SNPs
is probably causal
```

but the data cannot distinguish them perfectly.

---

### Visualizing SuSiE Results

The workshop visualizes:

```r
susie_plot(
    susie_results,
    y = "PIP"
)
```

Each point represents a SNP.

The y-axis shows:

```text 
Posterior Inclusion Probability
```

High peaks indicate likely causal variants.



---

### When Does SuSiE Work Best?

SuSiE performs best when:

### Large sample size

```text 
More information
```

---

### Lower LD

```text
Better separation
of causal variants
```

---

### Strong effects

```text 
Higher PIPs
```

---

### Accurate LD matrix

```text 
Correct ancestry matching
```

---

### Common Mistake

Many researchers assume:

```text 
The lead SNP
is causal.
```

SuSiE often demonstrates that:

```text 
Lead SNP ≠ Causal SNP
```

because the association signal may be distributed across correlated variants.

---

### SuSiE Versus ABF

| Feature                 | ABF     | SuSiE |
| ----------------------- | ------- | ----- |
| Bayesian                | Yes     | Yes   |
| Uses Summary Statistics | Yes     | Yes   |
| Uses LD Matrix          | No      | Yes   |
| Multiple Signals        | No      | Yes   |
| Credible Sets           | Limited | Yes   |
| Modern Standard         | No      | Yes   |

---

### Why SuSiE Became Popular

SuSiE provides:

* Accurate PIPs
* Multiple signal detection
* Credible sets
* Computational efficiency
* Summary-statistics support
* Compatibility with colocalization

For these reasons, it has become one of the most widely used fine-mapping methods in modern GWAS studies.

---

### Key Takeaways

SuSiE (Sum of Single Effects) is a Bayesian fine-mapping method designed to identify multiple causal variants within a genomic region. Unlike ABF, which assumes only one causal SNP per locus, SuSiE models several independent signals simultaneously. It requires GWAS summary statistics together with an LD matrix and produces Posterior Inclusion Probabilities (PIPs) and Credible Sets for each signal. SuSiE is currently one of the most popular fine-mapping methods because it provides accurate causal variant prioritization while accommodating the complexity of real genetic loci.

---


### Part 5: Running a Complete Fine-Mapping Analysis Step-by-Step

So far we have learned:

* What fine-mapping is
* Why GWAS cannot identify causal variants
* How ABF works
* How SuSiE improves upon ABF

Now it is time to walk through a complete fine-mapping workflow similar to the ISG workshop practical. We will:

```text
Simulate Genotypes
       ↓
Simulate Phenotype
       ↓
Run GWAS
       ↓
Run ABF
       ↓
Run SuSiE
       ↓
Interpret PIPs
       ↓
Interpret Credible Sets
```

This is essentially the workflow followed in many modern GWAS fine-mapping studies. 



---

### Step 1: Import Required Packages

```r
library(MASS)
library(susieR)
library(ggplot2)
```

Package roles:

```text
MASS      → Data simulation
susieR    → Fine-mapping
ggplot2   → Visualization
```

---

### Step 2: Simulate a Genetic Region

To understand fine-mapping, we first create a synthetic genomic region where we know the true causal variants.

Suppose:

```r
p <- 100
```

meaning:

```text
100 SNPs
```

in our locus.

---

### Create an LD Matrix

The workshop simulates a region where all SNPs have moderate correlation. 

```r
p <- 100

LD <- matrix(
  0.3,
  nrow = p,
  ncol = p
)

diag(LD) <- 1
```

This produces:

```text
Off-diagonal = 0.3
Diagonal = 1
```

representing moderate LD.

---

### Visualize the LD Matrix

```r
ld_df <- expand.grid(
  SNP1 = 1:p,
  SNP2 = 1:p
)

ld_df$LD <- as.vector(LD)

ggplot(
  ld_df,
  aes(
    SNP1,
    SNP2,
    fill = LD
  )
) +
geom_raster() +
coord_fixed()
```

Interpretation:

```text
Red = Strong LD

White = Weak LD
```

---

### Step 3: Simulate Genotypes

Now generate genotypes for:

```r
N <- 1000
```

individuals.

```r
set.seed(4)

X <- mvrnorm(
  n = N,
  mu = rep(0,p),
  Sigma = LD
)
```

Dimensions:

```r
dim(X)
```

Output:

```text
1000 × 100
```

Meaning:

```text
1000 individuals
100 SNPs
```

---

### Step 4: Define Causal Variants

Suppose:

```r
L <- 3
```

meaning:

```text
3 true causal SNPs
```

Select them randomly.

```r
set.seed(1)

causal_ind <- sample(
  1:p,
  L,
  replace = FALSE
)

causal_ind
```

Example output:

```text
68
39
1
```

These are the ground-truth causal variants.

The workshop uses this exact setup. 

---

### Step 5: Assign Effect Sizes

Assume:

```r
h2g <- 0.1
```

or:

```text
10% SNP heritability
```

Assign effects.

```r
per_snp_h2g <- h2g / L

effect_sizes <- rnorm(
  L,
  mean = 0,
  sd = sqrt(per_snp_h2g)
)
```

Example:

```text
Variant 68 → 0.18

Variant 39 → -0.23

Variant 1 → 0.14
```

---

### Step 6: Construct the Genetic Effect

Create beta coefficients.

```r
beta <- rep(0,p)

beta[causal_ind] <- effect_sizes
```

Calculate the genetic component.

```r
genetic_effect <- X %*% beta
```

---

### Step 7: Simulate Phenotypes

Add environmental noise.

```r
var_g <- var(genetic_effect)

sigma_squared <- ifelse(
  1-var_g > 0.05,
  1-var_g,
  0.1
)

epsilon <- rnorm(
  N,
  mean = 0,
  sd = sqrt(sigma_squared)
)

y <- as.numeric(
  genetic_effect + epsilon
)
```

Now we have:

```text
Genotypes
Phenotype
```

and can run GWAS.



---

### Step 8: Run GWAS

Create a results table.

```r
GWAS_df <- data.frame(
  beta = numeric(p),
  se = numeric(p),
  z = numeric(p),
  pval = numeric(p)
)
```

Loop through SNPs.

```r
for(i in 1:p){

  fit <- summary(
    lm(y ~ X[,i] - 1)
  )

  beta_hat <- fit$coefficients[
    1,
    "Estimate"
  ]

  se_hat <- fit$coefficients[
    1,
    "Std. Error"
  ]

  z <- beta_hat / se_hat

  p <- pchisq(
    z^2,
    df = 1,
    lower.tail = FALSE
  )

  GWAS_df[i,] <- c(
    beta_hat,
    se_hat,
    z,
    p
  )
}
```

This performs:

```text
100 separate GWAS tests
```

---

### Step 9: Visualize GWAS Results

```r
GWAS_df$SNP <- 1:nrow(GWAS_df)

GWAS_df$minus_log10p <-
  -log10(GWAS_df$pval)

ggplot(
  GWAS_df,
  aes(
    SNP,
    minus_log10p
  )
) +
geom_point()
```

This is a regional Manhattan plot.

The highest peaks represent candidate causal variants.

---

### Step 10: Apply ABF Fine-Mapping

Define the ABF function used in the workshop. 

```r
run_abf <- function(
  beta,
  stderr,
  W = 0.04
){

  z <- beta/stderr

  V <- stderr^2

  r <- W/(W+V)

  lbf <- (
    log(1-r) +
    r*(z^2)
  ) / 2

  lbf_max <- max(lbf)

  denom <- lbf_max +
    log(sum(
      exp(lbf-lbf_max)
    ))

  prob <- exp(
    lbf-denom
  )

  return(prob)
}
```

Run ABF.

```r
GWAS_df$PIP_ABF <-
  run_abf(
    GWAS_df$beta,
    GWAS_df$se
  )
```



---

### Step 11: Visualize ABF PIPs

```r
ggplot(
  GWAS_df,
  aes(
    SNP,
    PIP_ABF
  )
) +
geom_point()
```

Interpretation:

```text
Higher PIP
=
More likely causal
```

Often ABF will strongly prioritize only one signal.

---

### Step 12: Run SuSiE

Compute LD matrix.

```r
R <- cov(
  scale(X)
)
```

Run SuSiE.

```r
susie_results <- susie_rss(

  bhat = GWAS_df$beta,

  shat = GWAS_df$se,

  n = N,

  R = R,

  var_y = var(y),

  L = 4,

  estimate_residual_variance = TRUE
)
```

This mirrors the workshop analysis. 

---

### Step 13: Extract PIPs

```r
GWAS_df$PIP_SuSiE <-
  susie_results$pip
```

View top variants.

```r
head(
  GWAS_df[
    order(
      -GWAS_df$PIP_SuSiE
    ),
  ]
)
```

---



### Step 14: Plot SuSiE PIPs

```r
ggplot(
  GWAS_df,
  aes(
    SNP,
    PIP_SuSiE
  )
) +
geom_point()
```

You will often observe:

```text
Several distinct peaks
```

corresponding to multiple causal variants.



---

### Step 15: Examine Credible Sets

One of the most important outputs.

```r
susie_results$sets$cs
```

Example:

```text
$L1
39

$L2
20

$L3
68
```

Interpretation:

```text
Signal 1 → Variant 39

Signal 2 → Variant 20

Signal 3 → Variant 68
```



---

### Comparing ABF and SuSiE

ABF may identify:

```text
Variant 39
```

only.

SuSiE may identify:

```text
Variant 20

Variant 39

Variant 68
```

because it allows multiple causal variants.

This is precisely why SuSiE has become the preferred modern approach.



---

## Complete Workflow Summary

```text
Simulate Genotypes
       ↓
Simulate Phenotype
       ↓
GWAS
       ↓
ABF
       ↓
PIP Estimates
       ↓
SuSiE
       ↓
Multiple Signals
       ↓
Credible Sets
       ↓
Candidate Causal Variants
```

---

### Key Takeaways

A complete fine-mapping analysis begins with GWAS summary statistics and an LD matrix. ABF converts GWAS statistics into posterior probabilities under the assumption of a single causal variant. SuSiE extends this framework by modeling multiple independent signals within the same locus. The primary outputs are Posterior Inclusion Probabilities (PIPs) and Credible Sets, which identify the most likely causal variants. This workflow forms the foundation for downstream analyses such as eQTL mapping and colocalization.

---

### Part 6: Understanding Credible Sets in Practice

After running a fine-mapping method such as SuSiE, researchers often focus immediately on the Posterior Inclusion Probabilities (PIPs).

However, the most important output is usually:

### Credible Sets

Credible sets are the primary mechanism through which fine-mapping methods quantify uncertainty about causal variants.

Understanding credible sets properly is essential because most real GWAS loci do **not** resolve to a single variant.

The ISG workshop places substantial emphasis on interpreting credible sets and understanding why they may or may not contain the true causal variant. 

---







### Why Do We Need Credible Sets?

Suppose SuSiE reports:

```text
rs39
PIP = 0.98
```

Life is easy.

We can confidently prioritize:

```text
rs39
```

for downstream experiments.

Unfortunately, most loci look more like this:

```text
rs39  PIP = 0.32

rs40  PIP = 0.28

rs41  PIP = 0.22

rs42  PIP = 0.10

rs43  PIP = 0.08
```

Now we face uncertainty.

Which SNP is truly causal?

Instead of pretending to know the answer, Bayesian fine-mapping explicitly represents this uncertainty using credible sets.

---


### What Is a Credible Set?

A credible set is a collection of variants that jointly contain the causal variant with a specified probability.

Most commonly:

```text
95% Credible Set
```


Meaning:

```text
There is a 95% probability
that the causal SNP lies
within this set.
```

Notice the wording.

We are not saying:

```text
95% probability
for every SNP.
```

We are saying:

```text
95% probability
for the entire set.
```

---

### Constructing a Credible Set

Suppose SuSiE produces:

```text
rs39  PIP = 0.45

rs40  PIP = 0.25

rs41  PIP = 0.15

rs42  PIP = 0.08

rs43  PIP = 0.04

rs44  PIP = 0.03
```

Sort variants by decreasing PIP.

---

### Step 1

Add the largest PIP.

```text
rs39

Cumulative Probability = 0.45
```

Not enough.

---

### Step 2

Add the second variant.

```text
rs39 + rs40

0.45 + 0.25

= 0.70
```

Still not enough.

---

### Step 3

Add the next SNP.

```text
rs39 + rs40 + rs41

= 0.85
```

Still below 95%.

---

### Step 4

Add rs42.

```text
0.93
```

Still below threshold.

---

### Step 5

Add rs43.

```text
0.97
```

Now we exceed:

```text
95%
```

Therefore:

```text
95% Credible Set

=
{rs39, rs40, rs41, rs42, rs43}
```

---



### Interpreting a Credible Set

This does NOT mean:

```text
All five SNPs are causal.
```

Instead it means:

```text
One (or more)
of these SNPs
is likely causal,
but the data cannot
distinguish them.
```

---

### Ideal Credible Sets

The best possible outcome:

```text
95% Credible Set

=
{rs39}
```

Single-SNP credible set.

This indicates:

```text
Excellent resolution
```

and near-certain localization.

---

### Moderate Credible Sets

Example:

```text
95% Credible Set

=
{rs39, rs40, rs41}
```

This is still useful.

Experimental follow-up now requires testing:

```text
3 variants
```

instead of:

```text
500 variants
```

---

### Poor Credible Sets

Sometimes:

```text
95% Credible Set

=
50 SNPs
```

or even:

```text
100 SNPs
```

This indicates:

```text
Poor fine-mapping resolution
```

The data cannot isolate the causal variant.

---

### Why Do Large Credible Sets Occur?

Several factors contribute.




---

### Reason 1: High Linkage Disequilibrium

Suppose:

```text
rs39
rs40
rs41
rs42
```

all have:

```text
r² > 0.95
```

The data cannot distinguish them.

Result:

```text
Large Credible Set
```

---

### Reason 2: Small Sample Size

Imagine:

```text
N = 500
```

instead of:

```text
N = 50000
```

Less information is available.

PIPs become more diffuse.

Credible sets become larger.

The workshop explicitly demonstrates how increasing sample size improves fine-mapping resolution. 

---

### Reason 3: Weak Genetic Effects

Suppose:

```text
β = 0.02
```

instead of:

```text
β = 0.20
```

The signal becomes harder to localize.

Again:

```text
Larger Credible Sets
```









---

### Example: Excellent Fine-Mapping

```text
rs39  0.97

rs40  0.01

rs41  0.01

rs42  0.01
```

Credible Set:

```text
{rs39}
```

Interpretation:

```text
Near-perfect localization
```

---

### Example: Difficult Fine-Mapping

```text
rs39  0.25

rs40  0.24

rs41  0.22

rs42  0.16

rs43  0.13
```

Credible Set:

```text
{rs39, rs40, rs41, rs42, rs43}
```

Interpretation:

```text
Strong uncertainty
```

---

### Multiple Credible Sets

Modern loci often contain multiple signals.

Example:

```text
Signal 1
```

```text
CS1 = {rs20}
```

and

```text
Signal 2
```

```text
CS2 = {rs39, rs40}
```

and

```text
Signal 3
```

```text
CS3 = {rs68}
```

This is exactly what SuSiE was designed to discover.

Each credible set corresponds to a separate causal signal.

---


### Viewing Credible Sets in SuSiE

After fitting SuSiE:

```r
susie_results$sets$cs
```

Example output:

```text
$L1
39

$L2
20

$L3
68
```

Interpretation:

```text
Three independent signals
```

located at:

```text
Variant 39
Variant 20
Variant 68
```



The workshop uses this exact output structure. 

---

### Credible Set Coverage

Most studies report:

```text
95% coverage
```

but other thresholds are possible.

Examples:

```text
80%
90%
95%
99%
```

---

### Trade-Off

Higher coverage:

```text
99%
```

gives:

```text
More certainty
```

but:

```text
Larger Credible Sets
```

---

Lower coverage:

```text
80%
```

gives:

```text
Smaller Credible Sets
```

but:

```text
Greater risk
of missing the causal SNP
```



---

### The Workshop Example

The workshop explores changing credible set coverage from:

```text
95%
```

to

```text
80%
```

to observe how credible sets shrink as the confidence threshold decreases. 

---



### Credible Sets Versus Lead SNPs

Researchers often report:

```text
Lead SNP
```

which is simply:

```text
Smallest p-value
```

or

```text
Highest PIP
```

However:

```text
Lead SNP
≠
Credible Set
```

The credible set captures uncertainty.

The lead SNP does not.

This distinction is extremely important.

---


### What Should Be Reported in a Paper?

A modern fine-mapping study should report:

1. Lead SNP
2. PIP
3. Credible Set
4. Credible Set Size
5. Fine-mapping Method
6. LD Reference Panel

Example:

```text
Lead Variant:
rs39

PIP:
0.94

95% Credible Set:
{rs39, rs40}

Credible Set Size:
2
```

This provides much more information than a p-value alone.

---


### Biological Interpretation

Suppose:

```text
Credible Set Size = 1
```

Excellent candidate for:

* CRISPR validation
* Functional assays
* Reporter assays

---

Suppose:

```text
Credible Set Size = 50
```

Additional data may be required:

* Larger GWAS
* Better LD panel
* eQTL analysis
* Colocalization
* Functional annotations

to refine the signal.

---

### Key Takeaways

Credible sets are one of the most important outputs of modern fine-mapping methods. Rather than identifying a single causal variant, they quantify uncertainty by reporting a set of variants that collectively contain the causal SNP with a specified probability, typically 95%. Small credible sets indicate strong fine-mapping resolution, while large credible sets reflect uncertainty caused by high LD, limited sample size, or weak genetic effects. In modern statistical genetics, credible sets are often more informative than lead SNPs because they explicitly represent the uncertainty inherent in causal variant identification.

---



### Part 7: The Impact of Sample Size, Effect Size, and Linkage Disequilibrium on Fine-Mapping Resolution

One of the most important lessons in fine-mapping is:

```text
The quality of your fine-mapping results
depends heavily on your data.
```

Researchers often ask:

```text
Why is my credible set so large?

Why are my PIPs so low?

Why can't I identify the causal SNP?
```

The answer usually involves three factors:

1. Sample Size
2. Linkage Disequilibrium (LD)
3. Effect Size

The ISG workshop specifically explores these factors through simulation experiments. 

---



### Fine-Mapping Resolution

Resolution refers to how precisely we can localize a causal variant.

Excellent resolution:

```text
95% Credible Set

=
1 SNP
```

Poor resolution:

```text
95% Credible Set

=
50 SNPs
```

Our goal is always:

```text
Small Credible Sets
High PIPs
```

---

### Factor 1: Sample Size

The single most important factor in fine-mapping is:

```text
Sample Size
```

Suppose we conduct a GWAS with:

```text
N = 500
```

individuals.

The statistical signal will be noisy.

Now imagine:

```text
N = 500,000
```

The estimates become much more precise.

---

### Why Sample Size Matters

The standard error of a regression coefficient is approximately:

```text
SE ∝ 1/√N
```

Meaning:

```text
Larger N
     ↓
Smaller SE
     ↓
Larger Z-scores
     ↓
Higher PIPs
```

---

### Example

Suppose the true effect size is:

```text
β = 0.10
```

With:

```text
N = 500
```

we might observe:

```text
β = 0.10

SE = 0.08

Z = 1.25
```

Weak evidence.

---

With:

```text
N = 50,000
```

we might observe:

```text
β = 0.10

SE = 0.01

Z = 10
```

Very strong evidence.

---

### Simulation Example

The workshop increases the sample size from:

```text
N = 1000
```

to

```text
N = 5000
```

while keeping the same causal variants.

The result:

```text
Higher PIPs
Better localization
Smaller credible sets
```



---

### Visual Intuition

Small study:

```text
           *
        *  *
      *  * *
    * * * *
```

Noisy signal.

---

Large study:

```text
            *
            *
            *
            *
```

Sharper signal.

---

### Practical Rule

If you double the sample size:

```text
Information increases
```

If you increase sample size by:

```text
10×
```

fine-mapping performance often improves dramatically.

---

#### Factor 2: Linkage Disequilibrium (LD)

The second major factor is:

```text
LD
```

Fine-mapping is easiest when SNPs are weakly correlated.

---

### Low LD Example

```text
rs39 ---- rs40 ---- rs41
```

Correlations:

```text
r² = 0.05
```

Each SNP behaves independently.

Fine-mapping can easily identify:

```text
rs39
```

if it is causal.

---

### High LD Example

```text
rs39 ---- rs40 ---- rs41
```

Correlations:

```text
r² = 0.99
```

Now:

```text
rs39
rs40
rs41
```

look almost identical statistically.

---

### Consequence

The model cannot distinguish:

```text
Which SNP is causal?
```

Result:

```text
Large Credible Sets
```

---

### Workshop Demonstration

The workshop increases LD from:

```text
0.3
```

to

```text
0.95
```

while keeping sample size fixed.

Result:

```text
PIPs decrease

Credible sets grow

Resolution worsens
```



---

### Why LD Is So Difficult

Suppose:

```text
rs39
```

is causal.

If:

```text
r²(rs39, rs40) = 0.99
```

then:

```text
rs40
```

will have almost the same GWAS statistics.

Example:

```text
rs39   p = 1×10⁻¹²

rs40   p = 2×10⁻¹²
```

Both look causal.

The data cannot distinguish them.

---

### Fine-Mapping Resolution Spectrum

Low LD:

```text
Credible Set

=
{rs39}
```

Excellent.

---

Moderate LD:

```text
Credible Set

=
{rs39, rs40, rs41}
```

Acceptable.

---

Very High LD:

```text
Credible Set

=
25 SNPs
```

Poor.

---

### Factor 3: Effect Size

The third important factor is:

```text
Effect Size
```

Some variants have strong biological effects.

Others have very small effects.

---

### Large Effect Variant

```text
β = 0.25
```

Produces:

```text
Large Z-score

High PIP
```

Often:

```text
Single-SNP Credible Set
```

---

### Small Effect Variant

```text
β = 0.01
```

Produces:

```text
Small Z-score

Diffuse PIPs
```

Result:

```text
Large Credible Set
```

---

### Genetic Architecture Matters

Traits differ dramatically.

Examples:

```text
LDL Cholesterol
```

often contains:

```text
Large effects
```

---

Examples:

```text
Educational Attainment
```

often contains:

```text
Very small effects
```

---

Consequently:

```text
Some traits are naturally
easier to fine-map.
```

---

### Heritability and Fine-Mapping

The workshop simulates traits with:

```text
h² = 0.10
```

and suggests reducing:

```text
h² = 0.01
```

to observe what happens. 

When heritability decreases:

```text
Signals weaken

PIPs decrease

Credible sets increase
```

---


### Interaction Between Factors

These factors do not operate independently.

---

### Best Case Scenario

```text
Large Sample Size

Low LD

Large Effects
```

Result:

```text
Tiny Credible Sets

High PIPs
```

---

### Worst Case Scenario

```text
Small Sample Size

High LD

Tiny Effects
```

Result:

```text
Huge Credible Sets

Low PIPs
```


---

### Real-World Example

Suppose a locus contains:

```text
50 SNPs
```

in strong LD.

Even with:

```text
N = 100,000
```

fine-mapping may still struggle.

Why?

Because LD creates a fundamental statistical limitation.

More data helps, but cannot completely eliminate the problem.

---




### Why Different Ancestries Matter

Different populations have different LD structures.

European ancestry:

```text
Long LD blocks
```

Some regions difficult to resolve.

---

African ancestry:

```text
Shorter LD blocks
```

Often provides better fine-mapping resolution.

This is one reason multi-ancestry fine-mapping has become increasingly popular.

---

### Simulation Function from the Workshop

The workshop creates a function:

```r
simulate_and_finemap(
    N_sim,
    LD_baseline
)
```

allowing researchers to change:

```text
Sample Size

LD

Heritability
```

and directly observe how PIPs and credible sets change. 

This is an excellent exercise for understanding fine-mapping behavior.

---

### Practical Lessons

When fine-mapping performs poorly, ask:

### Is the sample size large enough?

```text
More samples
=
Better resolution
```

---

### Is LD too high?

```text
High LD
=
Harder localization
```

---

### Are effect sizes too small?

```text
Small effects
=
Lower PIPs
```

---

### Is the LD reference panel appropriate?

```text
Wrong ancestry
=
Potentially misleading results
```



---

### Common Beginner Mistake

Many people assume:

```text
Fine-mapping failed.
```

when they see:

```text
Credible Set Size = 50
```

In reality:

```text
The method may be working perfectly.
```

The data simply may not contain enough information to distinguish the variants.

This is an important mindset shift.

Fine-mapping quantifies uncertainty.

It does not magically eliminate uncertainty.

---

### Key Takeaways

Fine-mapping resolution is determined primarily by sample size, linkage disequilibrium, and effect size. Larger sample sizes produce more precise effect estimates and higher PIPs. High LD makes neighboring variants statistically indistinguishable and often leads to larger credible sets. Strong genetic effects are easier to localize than weak effects. Even the best fine-mapping method cannot overcome severe LD or limited information. Understanding these factors is essential for interpreting fine-mapping results realistically and avoiding overconfidence in causal variant claims.

---


### Part 8: Fine-Mapping with Individual-Level Data vs Summary Statistics

One of the first questions in any fine-mapping project is:

```text
What data do I actually have?
```

The answer determines:

* Which fine-mapping method you can use
* How accurate your results may be
* How LD should be estimated
* Whether colocalization is possible
* Which branch of the fine-mapping workflow you should follow

This is exactly the starting point of the ISG fine-mapping decision tree.

---

### Why Data Availability Matters

In an ideal world, every researcher would have:

```text
Genotypes
Phenotypes
Covariates
```

for every participant.

Unfortunately, this is rarely the case.

Most published GWAS provide only:

```text
Summary Statistics
```

such as:

```text
SNP
Beta
SE
P-value
Alleles
```

As a result, modern fine-mapping methods must work with multiple types of data.

---

### Two Main Types of Data

### Individual-Level Data

You have access to:

```text
Genotypes
Phenotypes
Covariates
```

for each person.

Example:

| Person | SNP1 | SNP2 | SNP3 | Phenotype |
| ------ | ---- | ---- | ---- | --------- |
| 1      | 0    | 1    | 2    | 3.2       |
| 2      | 1    | 1    | 0    | 1.7       |
| 3      | 2    | 0    | 1    | 4.5       |

---

### Summary Statistics

You only have GWAS results.

Example:

| SNP | Beta | SE   | P-value |
| --- | ---- | ---- | ------- |
| rs1 | 0.12 | 0.03 | 1e-8    |
| rs2 | 0.10 | 0.03 | 5e-7    |
| rs3 | 0.02 | 0.04 | 0.40    |

**No individual genotypes are available.**
---

### Why Summary Statistics Became Popular

Large GWAS datasets are often restricted.

Examples:

* UK Biobank
* FinnGen
* deCODE
* iPSYCH

Sharing raw genotypes creates:

* Privacy concerns
* Storage challenges
* Regulatory issues

Therefore most studies release:

```text
Summary Statistics
```

instead.

---

### Individual-Level Data Workflow

Suppose you have:

```text
Genotypes
Phenotypes
```

The workflow becomes:

```text
Raw Genotypes
      ↓
Quality Control
      ↓
GWAS
      ↓
Compute LD Directly
      ↓
Fine-Mapping
```

This is the gold standard.

---


### Why Individual-Level Data Is Powerful

Because you can calculate:

```text
LD
```

directly from the study participants.

Example:

```r
R <- cov(
  scale(X)
)
```

where:

```text
X
```

is the genotype matrix.

This produces:

```text
In-sample LD
```

which is the most accurate LD possible.

### What Is In-Sample LD?

Suppose your GWAS contains:

```text
10,000 Finnish individuals
```

You calculate LD using exactly those individuals.

Result:

```text
Perfectly matched LD
```

This is called:

```text
In-Sample LD
```

and is considered the gold standard.




### Summary Statistics Workflow

Now suppose you only have:

```text
Beta
SE
P-values
```

The workflow becomes:

```text
Summary Statistics
          ↓
External LD Panel
          ↓
Fine-Mapping
```

Now LD must come from somewhere else.

---


### External LD Reference Panels

Common reference panels include:

* 1000 Genomes
* UK Biobank
* TOPMed
* HRC

These provide genotype information for estimating LD.

---

### Example

Suppose your GWAS contains:

```text
European ancestry individuals
```

You may estimate LD from:

```text
1000 Genomes Europeans
```

instead.

---


### Why LD Matching Matters

This is one of the most important concepts in fine-mapping.

Suppose:

```text
GWAS
```

contains:

```text
Finnish ancestry
```

but LD comes from:

```text
African ancestry
```

Then:

```text
LD structure differs
```

which can distort fine-mapping results.


---

### Example

True GWAS LD:

```text
rs39 ↔ rs40

r² = 0.95
```

Reference LD:

```text
rs39 ↔ rs40

r² = 0.30
```

These tell completely different stories.

The fine-mapping method may become confused.

---

### Consequences of LD Mismatch

Poorly matched LD can cause:

```text
False signals
```

```text
Incorrect PIPs
```

```text
Wrong credible sets
```

```text
Failed colocalization
```

This is one of the most common causes of unreliable fine-mapping.



---

### Why SuSiE Requires Good LD

Recall that SuSiE models:

```text
Multiple causal variants
```

To separate these signals, it must know:

```text
Which SNPs are correlated?
```

Therefore:

```text
Good LD
=
Good Fine-Mapping
```

The workshop repeatedly emphasizes using accurate in-sample LD whenever possible. 




---

### Methods That Need LD

Examples:

```text
SuSiE
FINEMAP
CAVIAR
CARMA
```

All require LD information.

Without LD they cannot properly model correlation among SNPs.

---




### Methods That Can Work Without LD

Some methods use only summary statistics.

Example:

```text
ABF
```

The workshop ABF example requires:

```text
Beta
SE
```

but not an LD matrix. 

This is one reason ABF remains useful.

---



### Individual-Level Data Branch

The workshop flowchart recommends:

```text
Individual-Level Data
          ↓
Single Cohort
          ↓
Compute In-Sample LD
          ↓
SuSiE
```

This is typically the preferred route.

---

### Summary Statistics Branch

If only GWAS summary statistics are available:

```text
Summary Statistics
          ↓
Reference LD
          ↓
SuSiE
FINEMAP
CARMA
```

depending on the study design.



---

### Single-Cohort GWAS

Example:

```text
FinnGen
```

or

```text
UK Biobank
```

analyzed independently.

Workflow:

```text
Single Cohort
       ↓
Ancestry-Matched LD
       ↓
Fine-Mapping
```

This is generally straightforward.



---

### Meta-Analysis GWAS

Now consider:

```text
European Cohort
+
Finnish Cohort
+
Japanese Cohort
+
African Cohort
```

combined into one GWAS.

This creates a challenge.

---

### Why Meta-Analysis Is Difficult

Each cohort has:

```text
Different LD
```

Therefore:

```text
One LD Matrix
```

may no longer represent the data accurately.

This is one reason specialized meta-analysis fine-mapping methods were developed.

---



### Cohort-Specific Fine-Mapping

A common strategy is:

```text
Fine-map each cohort separately
```

then compare results.

Workflow:

```text
Cohort 1
      ↓
Fine-Mapping

Cohort 2
      ↓
Fine-Mapping

Cohort 3
      ↓
Fine-Mapping
```

Then integrate results.

---

### Multi-Ancestry Fine-Mapping

An increasingly popular approach is:

```text
Multi-Ancestry Fine-Mapping
```

because different ancestries have different LD structures.

Example:

European LD:

```text
rs39 rs40 rs41 rs42
```

all highly correlated.

---

African LD:

```text
rs39
```

more clearly separated.

Combining information can dramatically improve resolution.


---

### Practical Example

Imagine:

```text
European GWAS
```

produces:

```text
Credible Set Size = 40
```

African GWAS:

```text
Credible Set Size = 5
```

The African LD structure may help isolate the causal variant.

---


### Which Data Type Is Best?

Ranking:

### Best

```text
Individual-Level Data
+
In-Sample LD
```

---

### Good

```text
Summary Statistics
+
Matched LD Panel
```

---

### Risky

```text
Summary Statistics
+
Poorly Matched LD
```

---

### Worst

```text
Summary Statistics
+
No LD Information
```


---

### Practical Advice

Whenever possible:

### Use in-sample LD

```text
Best choice
```

---

### Match ancestry

```text
European GWAS
→ European LD
```

```text
African GWAS
→ African LD
```

---

### Avoid mixing populations

unless using methods designed for multi-ancestry analyses.

---


### Connection to Colocalization

Everything we learn here becomes even more important later.

Methods such as:

```text
coloc.susie
```

require accurate fine-mapping results.

Poor LD estimation can lead to:

```text
False colocalization
```

or

```text
Missed colocalization
```

Therefore:

```text
Good LD
=
Good Fine-Mapping
=
Good Colocalization
```

---

### Key Takeaways

The type of data available determines the entire fine-mapping strategy. Individual-level data provide the highest-quality analyses because LD can be computed directly from the study participants. When only summary statistics are available, an external LD reference panel must be used. Accurate ancestry matching between the GWAS cohort and the LD reference panel is critical because LD mismatch can distort PIPs, credible sets, and downstream analyses. Modern methods such as SuSiE, FINEMAP, and CARMA depend heavily on accurate LD information, making LD quality one of the most important determinants of successful fine-mapping.

---

### Part 9: Major Fine-Mapping Methods — ABF, SuSiE, FINEMAP, CAVIAR, CARMA, and FastMap

By now, you understand the core concepts behind fine-mapping:

* Linkage Disequilibrium (LD)
* Posterior Inclusion Probabilities (PIPs)
* Credible Sets
* Bayesian Inference
* Multiple Causal Variants

The next question naturally becomes:

```text
Which fine-mapping method should I use?
```

Modern statistical genetics offers several methods, each designed for different situations.

The ISG workshop flowchart includes methods such as:

```text
ABF
SuSiE
FINEMAP
CARMA
FastMap
```



Choosing the correct method depends largely on:

```text
Available Data
LD Information
Study Design
Computational Resources
```

---

### The Evolution of Fine-Mapping Methods

Historically:

```text
GWAS
   ↓
ABF
   ↓
CAVIAR
   ↓
FINEMAP
   ↓
SuSiE
   ↓
CARMA
   ↓
FastMap
```

Each generation addressed limitations of earlier methods.



---

### 1. Approximate Bayes Factor (ABF)

### Main Idea

Assume:

```text
One causal variant
per locus
```

Compute:

```text
Bayes Factors
```

for each SNP.

Convert them into:

```text
Posterior Probabilities
```

---

### Inputs

```text
Beta
SE
```

Only GWAS summary statistics are required.

No LD matrix is needed.

---

### Advantages

```text
Fast
Simple
Minimal inputs
```

---

### Disadvantages

```text
Single causal variant assumption
```

Cannot properly model:

```text
Multiple signals
```

---

### Best Use Case

When:

```text
No trustworthy LD matrix exists
```

or

```text
Quick exploratory analysis
```

---

### Example

Methods based on ABF:

```text
Wakefield ABF

coloc.abf
```

---

### 2. CAVIAR

One of the earliest methods designed to handle:

```text
Multiple causal variants
```

---

### Main Idea

Instead of assuming:

```text
1 causal SNP
```

CAVIAR models:

```text
Several causal SNPs
```

simultaneously.

---

### Inputs

```text
Z scores
LD matrix
```

---

### Outputs

```text
Causal probabilities

Credible sets
```

---

### Advantages

Accounts for LD.

Allows multiple causal variants.

---

### Disadvantages

Can become computationally expensive.

Performance decreases in very large regions.

---

### Historical Importance

CAVIAR was one of the first major advances beyond ABF.

Many later methods were inspired by it.

---

### 3. FINEMAP

FINEMAP became popular because it is:

```text
Fast
Accurate
Scalable
```

for large GWAS datasets.

---

### Main Idea

Uses Bayesian model selection.

Searches through possible causal configurations.

Example:

```text
Model 1:
rs39
```

```text
Model 2:
rs20 + rs39
```

```text
Model 3:
rs20 + rs39 + rs68
```

and evaluates which model best explains the data.

---

### Inputs

```text
Summary Statistics
LD Matrix
```

---

### Outputs

```text
PIPs

Credible Sets

Causal Configurations
```

---

### Advantages

Handles:

```text
Multiple causal variants
```

efficiently.

Widely used.

Very accurate.

---

### Disadvantages

Requires:

```text
Good LD
```

Performance may degrade if LD is poorly estimated.

---

### Best Use Cases

Large GWAS.

High-resolution fine-mapping.

Biobank-scale studies.

---

### 4. SuSiE (Sum of Single Effects)

SuSiE has become one of the most widely used methods in modern statistical genetics.

---

### Main Idea

Model phenotype as:

```text
Signal 1
+
Signal 2
+
Signal 3
```

rather than:

```text
One signal
```

---

### Inputs

```text
Summary Statistics

LD Matrix
```

or

```text
Individual-level data
```

---

### Outputs

```text
PIPs

Credible Sets
```

for each independent signal.

---

### Advantages

Excellent handling of:

```text
Multiple signals
```

Produces interpretable credible sets.

Computationally efficient.

Works well with colocalization.

---

### Disadvantages

Requires:

```text
Accurate LD
```

Sensitive to LD mismatch.

---

### Why SuSiE Is Popular

SuSiE became popular because it combines:

```text
Accuracy
Interpretability
Speed
```

while naturally handling multiple causal variants.

---

### 5. CARMA

CARMA stands for:

```text
CAusal Variants Identification
in the Presence of Allelic Heterogeneity
and Outliers
```

CARMA is a relatively recent method.

---

### Motivation

Traditional methods often assume:

```text
Summary statistics
contain no major errors.
```

Real GWAS may contain:

```text
Outlier SNPs

Genotyping artifacts

Imputation errors
```

These can distort fine-mapping.

---

### Main Idea

CARMA explicitly models:

```text
Outliers
```

and attempts to remain robust when data quality is imperfect.

---

### Inputs

```text
Summary Statistics

LD Matrix
```

---

### Advantages

Robust to:

```text
Outlier SNPs
```

Works well in challenging datasets.

Often improves credible set quality.

---

### Disadvantages

More computationally intensive.

More complex to interpret.

---

### Best Use Cases

When:

```text
Data quality is uncertain
```

or

```text
Large meta-analysis studies
```

where artifacts may occur.

---

### 6. FastMap

FastMap is one of the newest methods appearing in modern fine-mapping workflows.






---

### Motivation

Biobank-scale datasets now contain:

```text
Millions of individuals
```

and

```text
Thousands of loci
```

Traditional methods become computationally expensive.

---

### Main Idea

Use efficient approximations to dramatically speed up fine-mapping.


---

### Advantages

```text
Very fast
Scalable
```

Suitable for:

```text
Thousands of loci
```

simultaneously.

---

### Disadvantages

Still relatively new.

Less extensively validated than SuSiE or FINEMAP.





---

### Best Use Cases

Large-scale biobank projects.

Meta-analysis fine-mapping.

Population-scale studies.

---

### Comparison Table

| Method  | Multiple Signals | Needs LD | Speed     | Modern Usage |
| ------- | ---------------- | -------- | --------- | ------------ |
| ABF     | No               | No       | Very Fast | Moderate     |
| CAVIAR  | Yes              | Yes      | Moderate  | Historical   |
| FINEMAP | Yes              | Yes      | Fast      | High         |
| SuSiE   | Yes              | Yes      | Fast      | Very High    |
| CARMA   | Yes              | Yes      | Moderate  | Growing      |
| FastMap | Yes              | Yes      | Very Fast | Emerging     |

---

### Which Method Should You Use?

#### Scenario 1

You only have:

```text
Beta
SE
```

No LD matrix.

Use:

```text
ABF
```

---

### Scenario 2

You have:

```text
Summary Statistics
+
Good LD
```

Use:

```text
SuSiE
```

or

```text
FINEMAP
```



---

### Scenario 3

You suspect:

```text
Outliers
```

Use:

```text
CARMA
```


---

### Scenario 4

You need:

```text
Very large-scale analysis
```

Use:

```text
FastMap
```

---


### Scenario 5

You plan to perform:

```text
Colocalization
```

Use:

```text
SuSiE
```

because it integrates naturally with:

```text
coloc.susie
```

---


### The Current Consensus

If you ask statistical geneticists today:

```text
What is the default fine-mapping method?
```

the most common answer is:

```text
SuSiE
```

because it provides:

* Multiple signal modeling
* Credible sets
* PIPs
* Strong theoretical foundations
* Excellent software support

However:

```text
FINEMAP
```

remains extremely popular and is often used alongside SuSiE.

---

### A Practical Recommendation

For most researchers:

#### Individual-Level Data

```text
SuSiE
```


---

#### Summary Statistics + Good LD

```text
SuSiE or FINEMAP
```

---

### No LD Available

```text
ABF
```

---

#### Suspected Data Artifacts

```text
CARMA
```

---

#### Massive Biobank Analyses

```text
FastMap
```

---

### Fine-Mapping Methods in the Bigger Picture

All these methods aim to answer the same question:

```text
Which variants are most likely causal?
```


Their differences lie mainly in:

* Assumptions
* Computational strategy
* Treatment of multiple signals
* Robustness to imperfect data

The core outputs remain:

```text
Posterior Inclusion Probabilities

Credible Sets
```

which are the language of modern fine-mapping.

---

### Key Takeaways

Several fine-mapping methods are available, each designed for different scenarios. ABF is simple and useful when LD information is unavailable but assumes a single causal variant. CAVIAR was one of the first methods to model multiple causal variants. FINEMAP and SuSiE are currently among the most widely used methods because they efficiently model multiple signals and produce credible sets. CARMA extends fine-mapping by explicitly handling outlier variants, while FastMap focuses on computational scalability for biobank-sized datasets. In practice, SuSiE and FINEMAP have become the dominant choices for modern fine-mapping studies when reliable LD information is available.

---



### Part 10: The Fine-Mapping Decision Tree — Choosing the Correct Analysis Pipeline

One of the biggest challenges for beginners is not running fine-mapping software.

The real challenge is:

```text
Choosing the correct workflow.
```

Many researchers immediately ask:

```text
Should I use SuSiE?

Should I use FINEMAP?

Do I need CARMA?

Can I run coloc?

What LD matrix should I use?
```

The answer depends entirely on:

```text
What data are available.
```

This is why the ISG workshop introduced a Fine-Mapping Decision Tree.

Rather than choosing a method first, we should first determine:

```text
What information do we have?
```

---

### The Big Picture

Every fine-mapping project starts with:

```text
Available Data
       ↓
Appropriate LD
       ↓
Fine-Mapping Method
       ↓
Credible Sets
       ↓
Biological Interpretation
```

The workflow branches depending on the data type.

---

### Step 1: Do You Have Individual-Level Data?

The first question is:

```text
Do you have access to genotypes and phenotypes?
```

Examples:

```text
PLINK files

.bed
.bim
.fam
```

or

```text
VCF files

Phenotype files
```

If yes:

```text
Go down the
Individual-Level Data branch
```

If no:

```text
Go down the
Summary Statistics branch
```

---

### Branch A: Individual-Level Data

This is the ideal situation.

You have:

```text
Genotypes

Phenotypes

Covariates
```

Example datasets:

```text
UK Biobank

FinnGen

iPSYCH

All of Us
```

when individual-level access is granted.

---

### Why Individual-Level Data Is Valuable

Because you can calculate:

```text
In-Sample LD
```

directly from the study participants.

Workflow:

```text
Individual-Level Data
           ↓
Quality Control
           ↓
GWAS
           ↓
Compute In-Sample LD
           ↓
SuSiE
           ↓
Credible Sets
```

This is usually the preferred strategy.

---

### Single-Cohort Individual-Level Data

Suppose all samples come from:

```text
One Cohort
```

Example:

```text
FinnGen
```

or

```text
UK Biobank
```

alone.

Workflow:

```text
Single Cohort
      ↓
In-Sample LD
      ↓
SuSiE
```

This generally provides the most reliable fine-mapping.

---

### Multi-Cohort Individual-Level Data

Suppose you have:

```text
European Cohort

African Cohort

Asian Cohort
```

with full genotypes.

Now you have two options.

---

### Option 1: Fine-Map Separately

```text
European
     ↓
Fine-Map

African
     ↓
Fine-Map

Asian
     ↓
Fine-Map
```

Then compare results.

This is often recommended.

---

### Option 2: Multi-Ancestry Fine-Mapping

Combine all cohorts.

Use specialized approaches.

Advantages:

```text
Different LD structures
improve resolution
```

This has become increasingly popular in modern statistical genetics.

---

### Branch B: Summary Statistics Available

Most researchers fall into this category.

You have:

```text
Beta

SE

P-value
```

but not raw genotypes.

Workflow:

```text
Summary Statistics
           ↓
LD Reference Panel
           ↓
Fine-Mapping
```

---

### The Critical Question

The next question becomes:

```text
Do you have a good LD reference panel?
```

This is one of the most important decisions in fine-mapping.

---

### Scenario B1: Good LD Available

Example:

```text
European GWAS

European LD Panel
```

or

```text
Finnish GWAS

Finnish LD Panel
```

Excellent.

Workflow:

```text
Summary Statistics
          ↓
Matched LD
          ↓
SuSiE
or
FINEMAP
```

This is the most common workflow today.

---

### Why Matching Matters

Suppose:

```text
GWAS = European
```

but

```text
LD = African
```

Now:

```text
LD structure differs
```

which may distort:

```text
PIPs

Credible Sets
```

and potentially produce misleading results.

---

### Scenario B2: No Reliable LD Available

Sometimes you only have:

```text
Beta

SE
```

and no trustworthy LD.

In this situation:

```text
SuSiE
```

is not appropriate because it requires LD.

Instead:

```text
ABF
```

becomes useful.

Workflow:

```text
Summary Statistics
          ↓
ABF
          ↓
Posterior Probabilities
```

Remember:

```text
ABF assumes
one causal variant
per locus
```

so this is a compromise.

---

### Scenario B3: Suspected Data Problems

Sometimes summary statistics contain:

```text
Imputation errors

Outliers

Meta-analysis artifacts
```

These can distort fine-mapping.

Workflow:

```text
Summary Statistics
          ↓
LD Panel
          ↓
CARMA
```

CARMA was specifically developed to improve robustness to problematic variants.

---

### Meta-Analysis GWAS Branch

This is where things become complicated.

Suppose your GWAS combines:

```text
UK Biobank

FinnGen

deCODE

Biobank Japan
```

into a single meta-analysis.

Now:

```text
Multiple LD structures
```

exist simultaneously.

---

### Why Meta-Analysis Is Challenging

Fine-mapping methods assume:

```text
One LD matrix
```

but meta-analysis combines:

```text
Many LD matrices
```

This creates a mismatch.

---



### Strategy 1: Cohort-Specific Fine-Mapping

If cohort-specific summary statistics are available:

```text
FinnGen
     ↓
Fine-Map

UK Biobank
     ↓
Fine-Map

deCODE
     ↓
Fine-Map
```

Then integrate results.

This is often the preferred strategy.

---

### Strategy 2: Specialized Meta-Analysis Methods

If only meta-analysis results are available:

Use methods such as:

```text
FastMap
```

or related approaches designed for meta-analysis settings.

These methods attempt to account for heterogeneous LD structures.

---

### The Colocalization Branch

Eventually many projects move beyond fine-mapping.

Suppose you identify:

```text
GWAS Signal
```

and

```text
eQTL Signal
```

in the same region.

The next question becomes:

```text
Are they driven by
the same causal variant?
```

This is where:

```text
Colocalization
```
begins.

If you used ABF, you can use,

```text
coloc.abf
```

Workflow:

```text
GWAS Summary Statistics
            +
eQTL Summary Statistics
            ↓
coloc.abf
```

Assumption:

```text
One causal variant
per locus
```



 If You used SuSiE You can use:

```text
coloc.susie
```

Workflow:

```text
GWAS Fine-Mapping
          +
eQTL Fine-Mapping
          ↓
coloc.susie
```

This is generally preferred because it allows:

```text
Multiple causal variants
```

and aligns naturally with modern fine-mapping.

---

### The Complete Decision Tree

```text
Do you have individual-level data?
                │
       ┌────────┴────────┐
       │                 │
      Yes               No
       │                 │
       ↓                 ↓
Compute LD      Summary Statistics
       │                 │
       ↓                 ↓
   In-Sample LD   Good LD Available?
       │                 │
       │        ┌────────┴────────┐
       │        │                 │
       │       Yes               No
       │        │                 │
       ↓        ↓                 ↓
     SuSiE   SuSiE/FINEMAP      ABF
       │
       ↓
 Credible Sets
       │
       ↓
 Colocalization
       │
       ├── coloc.abf
       │
       └── coloc.susie
```

---

### Example 1: FinnGen Study

You have:

```text
FinnGen Summary Statistics
```

and

```text
FinnGen LD Panel
```

Recommendation:

```text
SuSiE
```

---

### Example 2: Public GWAS Catalog Result

You have:

```text
Beta

SE
```

but no LD information.

Recommendation:

```text
ABF
```

---

### Example 3: UK Biobank Genotypes

You have:

```text
Raw Genotypes
```

Recommendation:

```text
Compute In-Sample LD

Run SuSiE
```

---

### Example 4: Psychiatric Genomics Consortium Meta-Analysis

You have:

```text
Meta-analysis summary statistics
```

Recommendation:

```text
Cohort-specific fine-mapping
```

if possible.

Otherwise:

```text
FastMap
```

or another meta-analysis-aware approach.

---

### Common Beginner Mistake

Many people start with:

```text
Which method should I use?
```

The better question is:

```text
What data do I have?
```

The data determine the method.

Not the other way around.

---

### Recommended Default Workflow

For most modern studies:

```text
Summary Statistics
      +
Matched LD
      ↓
SuSiE
      ↓
Credible Sets
      ↓
eQTL Integration
      ↓
coloc.susie
```

This has become one of the most common pipelines in statistical genetics.

---

### Key Takeaways

The choice of fine-mapping method should be driven by the available data rather than personal preference. Individual-level data with in-sample LD provide the highest-quality analyses and are ideally suited for SuSiE. When only summary statistics are available, matched LD reference panels become critical. SuSiE and FINEMAP are generally preferred when reliable LD is available, while ABF remains useful when LD information is unavailable. Meta-analysis studies introduce additional complexity because LD differs across cohorts, often requiring cohort-specific fine-mapping or specialized methods such as FastMap. Ultimately, fine-mapping serves as the foundation for downstream analyses such as eQTL integration and colocalization.

---


### Part 11: Integrating Fine-Mapping with eQTL Data — Identifying the Target Gene Behind a GWAS Signal

At this point, we have learned how to:

```text
Run GWAS
      ↓
Fine-Map Loci
      ↓
Identify Candidate Causal Variants
```

However, a major biological question remains:

```text id="mns4g9"
Which gene is affected
by the causal variant?
```

Finding a causal SNP is only part of the story.

Ultimately, most researchers want to know:

```text id="76b7s3"
Which gene drives disease risk?
```

This is where eQTL analysis becomes one of the most powerful tools in statistical genetics.

---

### The Main Problem in GWAS

Most GWAS hits are:

```text id="65c65s"
Non-coding variants
```

rather than protein-coding mutations.

For example:

```text id="rpkfwy"
rs12345
```

may lie:

```text id="js9bpa"
50 kb upstream

or

100 kb downstream
```

of a gene.

The SNP itself does not tell us:

```text id="2m7cti"
Which gene is affected.
```

---





### What Is an eQTL?

An eQTL is a:

#### Expression Quantitative Trait Locus

A genetic variant that influences gene expression.

Example:

```text 
rs12345
```

may increase expression of:

```text 
Gene A
```

while decreasing expression of:

```text 
Gene B
```

---

### Conceptually

```text 
Genotype
      ↓
Gene Expression
      ↓
Disease Risk
```

This creates a biological mechanism linking DNA variation to disease.

---

### Why eQTLs Are Important

Suppose a schizophrenia GWAS identifies:

```text 
rs12345
```

Fine-mapping shows:

```text 
PIP = 0.95
```

Excellent.

But we still do not know:

```text 
Which gene
is responsible.
```

Now suppose:

```text 
rs12345
```

is also an eQTL for:

```text 
CACNA1C
```

This provides a biological hypothesis:

```text 
Variant
      ↓
CACNA1C Expression
      ↓
Disease Risk
```

---

### Where Do eQTL Data Come From?

Several large projects have measured gene expression and genotype data simultaneously.

Popular resources include:

* GTEx
* eQTL Catalogue
* PsychENCODE
* CommonMind Consortium
* eQTLGen

These datasets allow researchers to ask:

```text 
Does this SNP affect
gene expression?
```

---

### Cis-eQTLs

Most studies focus on:

#### Cis-eQTLs

Variants located near the gene they regulate.

Typically:

```text 
Within 1 Mb
```

of the gene.

Example:

```text 
rs12345
```

affects:

```text 
Gene A
```

located:

```text 
200 kb away
```

---

### Trans-eQTLs

Some variants regulate distant genes.

Example:

```text id="z1f92d"
Chromosome 1 SNP
```

affecting:

```text id="lmvm4n"
Chromosome 12 Gene
```

These are called:

#### Trans-eQTLs

and are generally harder to detect.

---

### The Fine-Mapping + eQTL Workflow

Modern statistical genetics often follows:

```text id="jlmq29"
GWAS
   ↓
Fine-Mapping
   ↓
Candidate SNPs
   ↓
eQTL Database
   ↓
Candidate Genes
```

---

### Example

Suppose fine-mapping produces:

```text id="d6mqt4"
95% Credible Set

=
rs39
rs40
rs41
```

Now we check eQTL databases.

Result:

```text id="4jyxya"
rs39 → Gene A

rs40 → No eQTL

rs41 → Gene B
```

Immediately:

```text id="4l6ylz"
Gene A
Gene B
```

become strong candidates.

---

### Why Fine-Mapping First?

Imagine starting with:

```text id="z6qxxd"
1000 GWAS SNPs
```

Checking all eQTLs would produce enormous numbers of candidate genes.

Fine-mapping reduces the search space:

```text id="l7v0pq"
1000 SNPs
      ↓
5 SNPs
```

making interpretation much easier.

---

### The Problem of Nearby Genes

A common beginner mistake is:

```text id="fpf8gb"
Nearest gene = causal gene
```

This is often incorrect.

Example:

```text id="tvvrfw"
GWAS SNP
```

may lie near:

```text id="ebg4ja"
Gene A
```

but actually regulate:

```text id="hzl9mk"
Gene B
```

located much farther away.

eQTL data help resolve this ambiguity.

---

### Fine-Mapping eQTL Signals

Interestingly, eQTL studies themselves can be fine-mapped.

Workflow:

```text id="ln09rl"
Expression Trait
        ↓
eQTL Mapping
        ↓
Fine-Mapping
        ↓
eQTL Credible Sets
```

This becomes important later when we discuss colocalization.

---

### Example: GWAS and eQTL Fine-Mapping

Suppose:

GWAS fine-mapping finds:

```text id="vcztuw"
CS1 = {rs39}
```

and eQTL fine-mapping finds:

```text id="0yxz6r"
CS1 = {rs39}
```

This is highly suggestive.

Both studies point to:

```text id="yxq9za"
rs39
```

as the causal variant.

---

### Biological Interpretation

Suppose:

```text id="yz67gf"
rs39
```

increases expression of:

```text id="z7l5e8"
Gene A
```

and also increases disease risk.

We may hypothesize:

```text id="h0axc8"
Higher Gene A Expression
      ↓
Higher Disease Risk
```

This provides a mechanistic explanation.

---

### Tissue Specificity

One of the most important aspects of eQTL analysis is:

#### Tissue Specificity

A variant may be an eQTL in:

```text id="y8pf7v"
Brain
```

but not in:

```text id="1nsz8d"
Blood
```

or

```text id="qfxr9j"
Liver
```

---

### Why Tissue Matters

Suppose we study:

```text id="k2f2p4"
Schizophrenia
```

Relevant tissues might include:

```text id="9s4fdh"
Prefrontal Cortex

Hippocampus

Neurons
```

Brain eQTLs are often more informative than blood eQTLs.

---

### Example: Psychiatric Genetics

Workflow:

```text id="srlhdi"
Schizophrenia GWAS
           ↓
Fine-Mapping
           ↓
PsychENCODE eQTLs
           ↓
Candidate Gene
```

This has become standard practice in psychiatric genomics.

---

### What Fine-Mapping Alone Cannot Tell Us

Fine-mapping identifies:

```text id="s1elxv"
Likely causal variants
```

but not necessarily:

```text id="n1mbjo"
Target genes
```

eQTL analysis identifies:

```text id="f1r91m"
Gene regulation
```

but not necessarily:

```text id="3gtx85"
Disease causality
```

Combining both provides much stronger evidence.

---

### Fine-Mapping + eQTL = Gene Prioritization

Together:

```text id="h9zttx"
Fine-Mapping
```

answers:

```text id="ig5uv0"
Which SNP?
```

while:

```text id="uknxxl"
eQTL Analysis
```

answers:

```text id="dtm8lv"
Which Gene?
```

---

### Remaining Problem

Even if:

```text id="sgw6gs"
GWAS SNP
```

and

```text id="hnp7lu"
eQTL SNP
```

appear similar,

we still cannot conclude:

```text id="aq5u80"
Same causal variant
```

because LD can create misleading overlap.

This is the exact motivation for:

#### Colocalization

which formally tests whether two association signals are driven by the same causal variant.

This is the subject of the next section.

---

### Example End-to-End Workflow

```text id="agkt65"
GWAS
   ↓
Fine-Mapping
   ↓
Credible Set
   ↓
eQTL Lookup
   ↓
Candidate Gene
   ↓
Colocalization
   ↓
Shared Causal Variant?
   ↓
Biological Mechanism
```

This workflow is now standard in many GWAS publications.

---

### Key Takeaways

Fine-mapping identifies likely causal variants, but it does not directly reveal which gene is affected. eQTL analysis provides the missing link by identifying variants that influence gene expression. By intersecting fine-mapped GWAS signals with eQTL results, researchers can prioritize candidate target genes and generate mechanistic hypotheses about disease biology. However, overlap between GWAS and eQTL signals does not automatically imply a shared causal variant because linkage disequilibrium can create misleading patterns. This limitation motivates the use of colocalization methods, which formally test whether two traits are driven by the same underlying variant.

---


---

### Part 12: Colocalization — Determining Whether a GWAS Signal and an eQTL Signal Share the Same Causal Variant

In the previous chapter, we learned how to use eQTL data to identify candidate target genes for a GWAS locus.

However, a critical problem remains.

Suppose we observe:

```text
GWAS Signal
```

and

```text
eQTL Signal
```

in the same genomic region.

Does that mean:

```text
Same Variant
```

causes both signals?

Unfortunately:

```text
No.
```

Not necessarily.

This is exactly why colocalization methods were developed.

The ISG workshop describes colocalization as the formal statistical test for determining whether two association signals are driven by the same underlying causal variant. 

---




### Why We Need Colocalization

Suppose a schizophrenia GWAS identifies:

```text
rs39
```

as a likely causal variant.

At the same locus, an eQTL study reports:

```text
rs40
```

associated with expression of:

```text
Gene A
```

Because:

```text
rs39 ↔ rs40
```

are in strong LD:

```text
r² = 0.95
```

both signals appear to overlap.

But two possibilities exist.

---

### Scenario 1: Shared Causal Variant

```text
rs39
     ↓
Gene A Expression

rs39
     ↓
Disease Risk
```

Same variant drives both traits.

This is true colocalization.

---

### Scenario 2: Distinct Variants

```text
rs39
     ↓
Disease Risk
```

and

```text
rs40
     ↓
Gene A Expression
```

Different variants.

Strong LD merely makes them appear similar.

This is NOT colocalization.

---

### The Goal of Colocalization

Colocalization asks:

```text
Do both traits
share the same
causal variant?
```

This is one of the most important questions in modern statistical genetics.

---

### The Five Colocalization Hypotheses

The coloc framework evaluates five mutually exclusive hypotheses. 

---

### H0

```text
No association
with either trait
```

Diagram:

```text
Trait 1   ✗

Trait 2   ✗
```

Nothing is happening.

---

### H1

```text
Association
with Trait 1 only
```

Diagram:

```text
Trait 1   ✓

Trait 2   ✗
```

Only the GWAS signal exists.

---

### H2

```text
Association
with Trait 2 only
```

Diagram:

```text
Trait 1   ✗

Trait 2   ✓
```

Only the eQTL signal exists.

---

### H3

```text
Both traits associated

Different causal variants
```

Diagram:

```text
Variant A
     ↓
Trait 1

Variant B
     ↓
Trait 2
```

This is often called:

```text
Linkage
```

or

```text
Distinct Signals
```

---

### H4

```text
Both traits associated

One shared
causal variant
```

Diagram:

```text
Variant A
     ↓
Trait 1

Variant A
     ↓
Trait 2
```

This is the outcome most researchers hope to see.

---

### Posterior Probabilities

coloc computes:

```text
PP.H0

PP.H1

PP.H2

PP.H3

PP.H4
```

These probabilities sum to:

```text
1
```

or

```text
100%
```

---

### Example

Suppose coloc reports:

```text
PP.H0 = 0.00

PP.H1 = 0.00

PP.H2 = 0.01

PP.H3 = 0.05

PP.H4 = 0.94
```

Interpretation:

```text
94% probability
of a shared
causal variant
```

Strong evidence for colocalization.

---

### Interpreting PP.H4

Typical interpretation:

```text
PP.H4 > 0.80
```

Strong evidence.

---

```text
PP.H4 > 0.90
```

Very strong evidence.

---

```text
PP.H4 > 0.95
```

Extremely strong evidence.

---

### Interpreting PP.H3

Suppose:

```text
PP.H3 = 0.95
```

Interpretation:

```text
Both traits
are associated

BUT

through different variants
```

This is a very different biological conclusion.

---

### coloc.abf

The original coloc method is:

#### coloc.abf

The ISG workshop describes it as a summary-statistics method requiring no LD matrix. 

---

### Inputs

```text
Beta

SE

Sample Size
```

for both traits.

Example:

```text
GWAS Summary Statistics

+

eQTL Summary Statistics
```

---

### Major Assumption

coloc.abf assumes:

```text
One causal variant
per locus
```

Exactly the same limitation as ABF fine-mapping.

---

### Why This Is a Problem

Real loci often contain:

```text
Signal 1

Signal 2

Signal 3
```

The workshop explicitly simulates a locus with multiple causal variants. 

In such situations:

```text
coloc.abf
```

can miss true colocalization.

---

### Example from the Workshop

Trait 1 (GWAS):

```text
Causal Variant 39
```

Trait 2 (eQTL):

```text
Variant 39
(shared)

Variant 20
(private)
```

The eQTL contains two signals. 

---

# Result

coloc.abf strongly favored:

```text
H3
```

meaning:

```text
Different variants
```

even though a shared variant truly existed. 

This demonstrates the limitation of the single-causal-variant assumption.

---

### coloc.susie

To solve this problem:

```text
coloc.susie
```

was developed.

The ISG workshop recommends it whenever reliable LD information is available. 

---

### Main Idea

Instead of comparing:

```text
One signal
```

against

```text
One signal
```

it compares:

```text
Credible Set 1

vs

Credible Set 1
```

```text
Credible Set 2

vs

Credible Set 2
```

and so on.

---

### Workflow

```text
GWAS
    ↓
SuSiE
    ↓
Credible Sets

eQTL
    ↓
SuSiE
    ↓
Credible Sets

        ↓

   coloc.susie
```

---

### Example Output

The workshop reports results like:

```text
GWAS Signal 39

eQTL Signal 39

PP.H4 = 1.0
```

meaning essentially perfect evidence for colocalization. 

---

### Why coloc.susie Is Preferred

Advantages:

```text
Multiple causal variants

Credible-set aware

Handles complex loci

Uses fine-mapping results
```

This makes it the modern standard.

---

### Biological Interpretation

Suppose:

```text
GWAS
```

colocalizes with

```text
Brain eQTL
```

for:

```text
CACNA1C
```

with:

```text
PP.H4 = 0.97
```

Interpretation:

```text
Strong evidence

that the same variant

influences both

CACNA1C expression

and disease risk
```

This creates a compelling mechanistic hypothesis.

---

### What Colocalization Does NOT Prove

Even a high:

```text
PP.H4
```

does NOT prove:

```text
Causality
```

It means:

```text
The data are consistent
with one shared variant
```

Additional biological validation is still required. 

---

### Key Takeaways

Colocalization is the statistical framework used to determine whether two association signals are driven by the same causal variant. The coloc framework evaluates five hypotheses (H0–H4), with H4 representing a shared causal variant and H3 representing distinct causal variants. The original coloc.abf method requires only summary statistics but assumes a single causal variant per locus. Modern analyses increasingly use coloc.susie because it leverages SuSiE fine-mapping results and can accommodate multiple causal variants. Colocalization serves as the critical bridge between GWAS findings and gene regulatory mechanisms, helping researchers connect disease-associated variants to specific genes and biological pathways.

---

### Part 13: Practical Interpretation of Colocalization Results — Reading PP.H0–PP.H4, Common Pitfalls, and Real-World Applications

Running a colocalization analysis is relatively easy.

Interpreting the results correctly is much harder.

Many beginners see:

```text
PP.H4 = 0.85
```

and immediately conclude:

```text
Gene identified!
```

Unfortunately, interpretation requires considerably more care.

In this chapter we will learn:

* How to interpret PP.H0–PP.H4
* What constitutes strong evidence
* Common mistakes
* Sensitivity analyses
* Real-world examples
* How fine-mapping quality affects colocalization

This is the final skill needed before applying colocalization in real studies.

---

### The Most Important Output

After running:

```r
coloc.abf(...)
```

or

```r
coloc.susie(...)
```

you typically obtain:

```text
PP.H0
PP.H1
PP.H2
PP.H3
PP.H4
```

These probabilities always sum to:

```text
1
```

---

### Interpretation Refresher

#### H0

```text
No association
with either trait
```

---

#### H1

```text
Trait 1 only
```

Example:

```text
GWAS signal exists

eQTL signal absent
```

---

#### H2

```text
Trait 2 only
```

Example:

```text
eQTL signal exists

GWAS signal absent
```

---

#### H3

```text
Both traits associated

Different causal variants
```

---

#### H4

```text
Both traits associated

Shared causal variant
```

This is usually the desired outcome.







---

### Example 1: Strong Colocalization

Suppose results are:

```text
PP.H0 = 0.00

PP.H1 = 0.01

PP.H2 = 0.02

PP.H3 = 0.03

PP.H4 = 0.94
```

Interpretation:

```text
Very strong evidence
for a shared causal variant
```

This is the textbook example of successful colocalization.

---

### Example 2: Strong Evidence Against Colocalization

Suppose:

```text
PP.H0 = 0.00

PP.H1 = 0.00

PP.H2 = 0.00

PP.H3 = 0.96

PP.H4 = 0.04
```

Interpretation:

```text
Both traits are associated

BUT

different causal variants
appear responsible
```

This is one of the most common outcomes.

---

### Example 3: Ambiguous Region

Suppose:

```text
PP.H0 = 0.05

PP.H1 = 0.10

PP.H2 = 0.10

PP.H3 = 0.35

PP.H4 = 0.40
```

Interpretation:

```text
Data are inconclusive
```

Neither:

```text
H3
```

nor

```text
H4
```

dominates.

Additional data may be needed.

---

### Practical Thresholds

Many studies use:

#### Weak Evidence

```text
PP.H4 < 0.50
```

---

#### Moderate Evidence

```text
0.50 ≤ PP.H4 < 0.80
```

---

#### Strong Evidence

```text
PP.H4 ≥ 0.80
```

---

#### Very Strong Evidence

```text
PP.H4 ≥ 0.90
```

---

#### Exceptional Evidence

```text
PP.H4 ≥ 0.95
```

These thresholds are conventions rather than strict rules.

Context always matters.

---

### Looking Beyond PP.H4

Many beginners focus only on:

```text
PP.H4
```

This is dangerous.

Consider:

```text
PP.H3 = 0.45

PP.H4 = 0.50
```

Although:

```text
PP.H4
```

is the largest probability,

there remains substantial support for:

```text
Different causal variants
```

The result is uncertain.



---

### The H3 Versus H4 Competition

In practice:

```text
H3
```

and

```text
H4
```

are often the most important hypotheses.

The real question becomes:

```text
Shared Variant?

or

Different Variants?
```

A useful diagnostic is:

```text
PP.H4 / (PP.H3 + PP.H4)
```

Large values indicate stronger evidence for sharing.

---


### Example

Suppose:

```text
PP.H3 = 0.10

PP.H4 = 0.90
```

Then:

```text
0.90 / (0.90 + 0.10)

= 0.90
```

Excellent support for sharing.

---

### Why Fine-Mapping Quality Matters

Colocalization depends on fine-mapping.

Poor fine-mapping leads to poor colocalization.

Example:

```text
GWAS Credible Set

=
50 SNPs
```

and

```text
eQTL Credible Set

=
60 SNPs
```

The uncertainty becomes enormous.

Colocalization may become inconclusive.


---

### High-Resolution Example

GWAS:

```text
{rs39}
```

eQTL:

```text
{rs39}
```

Result:

```text
PP.H4 ≈ 1
```

Much easier to interpret.

---



### Common Mistake #1

#### Assuming Overlap Equals Colocalization

Researchers sometimes observe:

```text
GWAS peak

and

eQTL peak
```

in the same region.

They conclude:

```text
Same signal
```

This is incorrect.

Strong LD can create apparent overlap even when different variants are responsible.

This is exactly why formal colocalization methods exist.

---



### Common Mistake #2

#### Ignoring LD Mismatch

Suppose:

```text
GWAS
```

uses:

```text
European samples
```

while:

```text
eQTL
```

uses:

```text
African samples
```

LD patterns differ.

Fine-mapping may identify different credible sets even when the biology is identical.

Always consider ancestry.


---

### Common Mistake #3

#### Assuming Colocalization Proves Causality

Even:

```text
PP.H4 = 0.99
```

does not prove:

```text
Gene A causes disease.
```

It only suggests:

```text
Shared genetic regulation.
```

Further evidence may be required:

* Functional studies
* CRISPR experiments
* Animal models
* Perturbation experiments

---



### Common Mistake #4

#### Ignoring Tissue Context

Suppose:

```text
Brain eQTL

PP.H4 = 0.95
```

and

```text
Blood eQTL

PP.H4 = 0.05
```

for schizophrenia.

The brain result is biologically more relevant.

Tissue selection matters enormously.


---

### Sensitivity Analysis

A good colocalization study should evaluate:

```text
Different priors
```

because Bayesian results depend on prior assumptions.

The coloc package allows prior specification.

Example:

```r
coloc.abf(
  dataset1,
  dataset2,
  p1 = 1e-4,
  p2 = 1e-4,
  p12 = 1e-5
)
```

Researchers often test multiple prior settings.

---

### What Are Priors?

Priors represent beliefs about:

```text
How likely a SNP is
to affect Trait 1
```

```text
How likely a SNP is
to affect Trait 2
```

```text
How likely a SNP affects both
```

Different assumptions can influence results.



---

### Example of Sensitivity

Suppose:

```text
PP.H4 = 0.82
```

under one prior.

and

```text
PP.H4 = 0.35
```

under another.

Interpretation:

```text
Result is unstable
```

Be cautious.

---

### Real-World Example Workflow

Consider:

```text
Depression GWAS
```

and

```text
Brain eQTL Data
```

Workflow:

```text
GWAS
  ↓
Fine-Mapping
  ↓
Credible Sets
  ↓
Brain eQTL
  ↓
Fine-Mapping
  ↓
Credible Sets
  ↓
coloc.susie
```

Result:

```text
PP.H4 = 0.92
```

Interpretation:

```text
Strong evidence

that the same variant

influences both

gene expression

and depression risk
```

This provides a biologically meaningful hypothesis.



---

### What Should Be Reported?

A publication should typically report:

```text
Gene

Tissue

PP.H4

PP.H3

Method Used

Credible Set Sizes

Fine-Mapping Method
```

Example:

```text
Gene:
CACNA1C

Tissue:
Prefrontal Cortex

Method:
coloc.susie

PP.H4:
0.96

GWAS CS Size:
2

eQTL CS Size:
1
```

This provides a complete picture.




---

### The Hierarchy of Evidence

Increasing confidence:

```text
GWAS Association
      ↓
Fine-Mapping
      ↓
eQTL Association
      ↓
Colocalization
      ↓
Functional Validation
```

Each step strengthens the evidence.

---


### Practical Checklist

Before trusting a colocalization result, ask:

#### Are both traits associated?

```text
Check H1/H2/H3/H4
```

---

#### Is PP.H4 high?

```text
Preferably > 0.8
```

---

#### Is PP.H3 low?

```text
Different-variant explanation
should be weak
```

---

#### Were credible sets small?

```text
Better resolution
```

---

#### Is LD appropriately matched?

```text
Critical
```

---

#### Is the tissue biologically relevant?

```text
Very important
```

---




### Were sensitivity analyses performed?

```text
Highly recommended
```

---

### Key Takeaways

Colocalization results should be interpreted using the full set of posterior probabilities rather than PP.H4 alone. Strong evidence for colocalization typically requires a high PP.H4 and a low PP.H3. Fine-mapping quality directly affects colocalization performance because poorly resolved credible sets introduce uncertainty. Colocalization does not prove causality but provides strong evidence that the same genetic variant influences both a molecular trait and a disease trait. Proper interpretation requires attention to ancestry matching, tissue relevance, prior assumptions, and sensitivity analyses.

---



### Part 14: End-to-End Statistical Genetics Pipeline — From GWAS to Biological Discovery

We have now reached the final chapter of this tutorial.

Throughout the previous chapters, we explored:

```text
GWAS
Fine-Mapping
eQTL Analysis
Colocalization
```

individually.

In practice, however, researchers do not perform these analyses in isolation.

Instead, they are combined into a single integrated workflow whose ultimate goal is:

```text
Move from

Association

to

Biological Mechanism
```

This chapter ties together everything we have learned into one coherent pipeline.

---

### The Fundamental Problem

Suppose a GWAS identifies:

```text
Chromosome 6
```

as associated with schizophrenia.

GWAS tells us:

```text
Something important
exists in this region.
```

But it does not tell us:

```text
Which variant?

Which gene?

Which mechanism?
```

This is the motivation behind the modern post-GWAS pipeline.

---



### The Complete Workflow

The modern statistical genetics workflow can be summarized as:

```text
Genotypes
      ↓
GWAS
      ↓
Significant Locus
      ↓
Fine-Mapping
      ↓
Credible Sets
      ↓
eQTL Integration
      ↓
Colocalization
      ↓
Target Gene
      ↓
Functional Validation
      ↓
Biological Mechanism
```

This framework underlies many contemporary studies from:

* UK Biobank
* FinnGen
* Psychiatric Genomics Consortium
* GTEx
* PsychENCODE

---

### Step 1: GWAS

Everything begins with a Genome-Wide Association Study.

Input:

```text
Genotypes

Phenotypes

Covariates
```

Example:

```text
500,000 individuals

10 million SNPs
```

Typical GWAS model:

```text
Phenotype

=

SNP

+

Covariates

+

Error
```

Output:

```text
Beta

SE

P-value
```

for every SNP.










---

### Example GWAS Result

Suppose:

```text
rs39

p = 1×10⁻¹²
```

This indicates:

```text
Strong association
```

but not causality.


---

### Limitation of GWAS

Because of LD:

```text
rs39
rs40
rs41
rs42
```

may all appear significant.

GWAS cannot determine:

```text
Which SNP
is causal.
```

This leads to the next step.

---



### Step 2: Fine-Mapping

Fine-mapping attempts to identify:

```text
Likely causal variants
```

rather than merely associated variants.

Inputs:

```text
GWAS Summary Statistics

+

LD Matrix
```

Methods:

```text
SuSiE

FINEMAP

CARMA

ABF
```

---

### Fine-Mapping Output

Example:

```text
rs39

PIP = 0.94
```

and

```text
95% Credible Set

=
{rs39, rs40}
```

Interpretation:

```text
The causal variant
is probably one of these SNPs.
```

---

### Why Fine-Mapping Matters

Without fine-mapping:

```text
100 SNPs
```

may require investigation.

After fine-mapping:

```text
2 SNPs
```

remain.

This dramatically improves biological interpretation.

---


### Step 3: Gene Prioritization

Now we ask:

```text
Which gene
is affected?
```

The nearest gene is often:

```text
Not
the causal gene.
```

Therefore we integrate functional genomics.

---





### Step 4: eQTL Analysis

An eQTL identifies variants affecting gene expression.

Example:

```text
rs39
```

influences:

```text
CACNA1C expression
```

in brain tissue.

This creates a hypothesis:

```text
rs39
     ↓
CACNA1C Expression
     ↓
Disease Risk
```

---

### Why eQTLs Are Useful

Fine-mapping answers:

```text
Which SNP?
```

eQTL analysis answers:

```text
Which gene?
```

Combining them dramatically improves interpretation.

---


### Step 5: Fine-Mapping the eQTL

Modern analyses often fine-map:

```text
GWAS
```

and

```text
eQTL
```

independently.

Example:

GWAS:

```text
CS = {rs39}
```

eQTL:

```text
CS = {rs39}
```

This is highly suggestive.

But still not proof.


---

### Step 6: Colocalization

Now we ask:

```text
Do both traits
share the same
causal variant?
```

This is the role of:

```text
coloc.abf

coloc.susie
```



---

### The Five Hypotheses

Colocalization evaluates:

```text
H0
No association
```

```text
H1
Trait 1 only
```

```text
H2
Trait 2 only
```

```text
H3
Different variants
```

```text
H4
Shared variant
```

---

### Example

Suppose:

```text
PP.H4 = 0.96
```

Interpretation:

```text
Strong evidence
for a shared
causal variant.
```

This is often the key result in modern post-GWAS studies.

---




### Putting It Together

Suppose:

GWAS finds:

```text
rs39
```

Fine-mapping finds:

```text
PIP = 0.98
```

eQTL analysis finds:

```text
rs39 affects CACNA1C
```

Colocalization finds:

```text
PP.H4 = 0.97
```

This supports:

```text
rs39
      ↓
CACNA1C Expression
      ↓
Schizophrenia Risk
```

A biologically meaningful hypothesis has emerged.

---

### What Have We Achieved?

Originally:

```text
10 million SNPs
```

were tested.

Now we have:

```text
1 variant

1 gene

1 mechanism
```

This is the power of the integrated pipeline.

---

### Real Example: Psychiatric Genetics

A common workflow in psychiatric genomics:

```text
Schizophrenia GWAS
          ↓
Fine-Mapping
          ↓
Brain eQTL Data
          ↓
Colocalization
          ↓
Candidate Gene
          ↓
Functional Validation
```

Resources often include:

```text
PsychENCODE

GTEx

CommonMind
```


---

### Beyond eQTLs

Modern studies increasingly integrate:

```text
sQTLs
```

(splicing QTLs)

```text
pQTLs
```

(protein QTLs)

```text
mQTLs
```

(methylation QTLs)

```text
caQTLs
```

(chromatin accessibility QTLs)

The same fine-mapping and colocalization framework applies.



---

### Multi-Omics Integration

A modern workflow may look like:

```text
GWAS
   ↓
Fine-Mapping
   ↓
eQTL Colocalization
   ↓
pQTL Colocalization
   ↓
Single-Cell Expression
   ↓
Pathway Analysis
   ↓
Drug Target Discovery
```

This represents the current frontier of statistical genetics.

---

### Where Single-Cell Data Fits

Recently, many studies have added:

```text
Single-Cell RNA-seq
```

to identify:

```text
Which cell types
express the target gene?
```

Example:

```text
Gene A
```

may be highly expressed in:

```text
Excitatory Neurons
```

but not:

```text
Microglia
```

providing additional biological context.



---

### Where TWAS Fits

Another increasingly popular method is:

```text
TWAS
```

Workflow:

```text
GWAS
   ↓
Predicted Expression
   ↓
Gene-Level Association
```

TWAS complements:

```text
Fine-Mapping

eQTL Analysis

Colocalization
```

and is often used together with them.


---

### Where Mendelian Randomization Fits

Another downstream step is:

```text
Mendelian Randomization
```

Workflow:

```text
eQTL
   ↓
Gene Expression
   ↓
Disease
```

MR can test:

```text
Is altered expression
causally related
to disease?
```

This provides stronger evidence than colocalization alone.




---

### The Hierarchy of Evidence

Increasing confidence:

```text
GWAS
      ↓
Fine-Mapping
      ↓
eQTL Association
      ↓
Colocalization
      ↓
TWAS
      ↓
Mendelian Randomization
      ↓
Functional Validation
```

Each layer adds evidence.

---

### The Ultimate Goal

The goal is not simply:

```text
Find significant SNPs
```

The real goal is:

```text
Understand biology
```

Specifically:

```text
Variant
     ↓
Gene
     ↓
Cell Type
     ↓
Pathway
     ↓
Disease
```

---




### Common Beginner Misconception

Many researchers stop after:

```text
GWAS
```

However:

```text
GWAS
```

is often just the starting point.

The real biological discoveries usually emerge during:

```text
Fine-Mapping

eQTL Integration

Colocalization

Functional Interpretation
```

---

### A Modern Statistical Geneticist's Workflow

Today, a typical project might involve:

```text
GWAS
```

↓

```text
SuSiE Fine-Mapping
```

↓

```text
GTEx eQTL Lookup
```

↓

```text
coloc.susie
```

↓

```text
TWAS
```

↓

```text
Single-Cell Annotation
```

↓

```text
Experimental Validation
```

This represents the modern state of the field.



---

### Final Key Takeaways

The journey from GWAS to biological discovery involves several interconnected analytical steps. GWAS identifies associated genomic regions, fine-mapping narrows these regions to likely causal variants, eQTL analysis links variants to gene expression, and colocalization tests whether the same variant influences both expression and disease risk. Together, these approaches transform statistical associations into biologically meaningful hypotheses. Modern statistical genetics increasingly integrates additional layers such as single-cell genomics, TWAS, Mendelian Randomization, and multi-omics datasets to build a comprehensive understanding of disease mechanisms. The ultimate objective is to move from a significant SNP to a causal gene, a relevant cell type, a biological pathway, and eventually therapeutic insight.

---

